# CrowdVision Campus — cloud GPU training
Improve head counting for recorded campus occupancy review using SCUT-HEAD Part B (academic research only).

Select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. CPU training is refused. No Drive mounting or local footage uploads are needed.

This notebook downloads the official dataset, converts its head boxes, excludes exact duplicate images across splits, and fine-tunes the existing YOLO11m detector. Validation data tunes confidence; the separate test split compares counting MAE/RMSE and detection precision/recall. Lower MAE is better, not an accuracy percentage.

The original checkpoint's training exposure is unknown. This comparison does not establish independence from its original training set or accuracy at a particular campus.

Download the output ZIP before disconnecting. Colab resources and runtime lifetime are limited.

Dataset source: https://github.com/HCIILAB/SCUT-HEAD-Dataset-Release


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.90', 'gdown==6.4.2', 'streamlit>=1.59,<2', 'reportlab>=4,<6', 'scipy>=1.12,<2', 'imageio-ffmpeg>=0.6,<1'], check=True)
import torch
assert torch.cuda.is_available(), 'No GPU allocated. Select Runtime > Change runtime type > T4 GPU.'
print('Cloud GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)


In [ ]:
from pathlib import Path
import json, os, hashlib, urllib.request
ROOT=Path('/content/crowdvision_gpu')
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)
SOURCES={'src/__init__.py': '"""\nsrc package initialization\n"""\n', 'src/core/detector.py': '"""\ndetect.py\n\nRuns head detection using a YOLO11 model (custom-trained head model, or\nthe pretrained person-detecting yolo11n.pt/yolo11s.pt with head-region\nestimation). Draws blue bounding boxes around detected heads, counts\nthem, classifies crowd level, and saves the annotated result.\n\nExposes `load_model()` and `detect_heads_in_frame()` so video_detect.py\ncan reuse the same detection logic per-frame without reloading the model\neach time. `run_detection()` (single image, unchanged signature) is kept\nfor backward compatibility with app.py\'s existing image-upload flow.\n\nUsage:\n    python detect.py --weights yolov8n.pt --source image.jpg\n    python detect.py --weights best.pt --source image.jpg --conf 0.4 --save-dir outputs\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport threading\n\n_INFERENCE_LOCK = threading.RLock()\nfrom typing import List, Optional\n\nimport cv2\nfrom ultralytics import YOLO  # noqa: F401 (re-exported / used as type hint below)\n\nimport config\nfrom src.utils.helpers import (\n    Detection,\n    draw_detections,\n    draw_overlay,\n    ensure_dir,\n    estimate_body_box_from_head,\n    estimate_head_box,\n    get_auto_crowd_level,\n    get_auto_risk_level,\n    get_crowd_level,\n    is_plausible_head_box,\n    is_valid_weights_reference,\n    resolve_class_ids,\n    setup_logger,\n)\n\nlogger = setup_logger(__name__)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Run YOLO11 head detection on a single image.")\n    parser.add_argument("--weights", type=str, default=config.BEST_WEIGHTS_PATH, help="Path to trained .pt weights")\n    parser.add_argument("--source", type=str, required=True, help="Path to input image")\n    parser.add_argument("--conf", type=float, default=config.INFERENCE_CONF_THRESHOLD, help="Confidence threshold")\n    parser.add_argument("--iou", type=float, default=config.INFERENCE_IOU_THRESHOLD, help="IoU threshold for NMS")\n    parser.add_argument("--imgsz", type=int, default=config.INFERENCE_IMGSZ, help="Inference image size")\n    parser.add_argument("--device", type=str, default=config.INFERENCE_DEVICE, help="Inference device")\n    parser.add_argument("--save-dir", type=str, default=config.OUTPUT_IMAGES_DIR, help="Directory to save result")\n    return parser.parse_args()\n\n\ndef load_model(weights_path: str) -> YOLO:\n    """Load a YOLO model once, for reuse across many frames (images or video)."""\n    if not is_valid_weights_reference(weights_path):\n        raise FileNotFoundError(f"Weights not found: {weights_path}")\n    return YOLO(weights_path)\n\n\ndef _model_has_native_head_class(model: YOLO) -> bool:\n    """True if the loaded model already has a dedicated \'head\' class."""\n    return resolve_class_ids(model.names, config.NATIVE_HEAD_CLASS_NAMES, allow_fallback=False) is not None\n\n\ndef detect_heads_in_frame(\n    model: YOLO,\n    frame,\n    conf: float = config.INFERENCE_CONF_THRESHOLD,\n    iou: float = config.INFERENCE_IOU_THRESHOLD,\n    imgsz: int = config.INFERENCE_IMGSZ,\n    device: str = config.INFERENCE_DEVICE,\n    class_ids: Optional[list] = None,\n    native_head_class: Optional[bool] = None,\n    show_body_box_estimate: bool = config.SHOW_BODY_BOX_ESTIMATE,\n) -> List[Detection]:\n    """\n    Run detection on a single frame (numpy BGR array) and return a list of\n    HEAD bounding boxes.\n\n    If the model has a native "head" class, its boxes are used directly.\n    Otherwise (e.g. pretrained yolo11n.pt/yolo11s.pt detecting "person"),\n    each person box is narrowed down to an estimated head region via\n    utils.estimate_head_box().\n    """\n    if native_head_class is None:\n        native_head_class = _model_has_native_head_class(model)\n    # A model containing both classes must not count a head and its person twice.\n    class_ids = resolve_class_ids(model.names, config.NATIVE_HEAD_CLASS_NAMES if native_head_class else (\'person\',))\n    if getattr(model, \'task\', \'detect\') != \'detect\':\n        raise ValueError(\'Choose an object-detection model with head or person classes.\')\n\n    # The cached model is shared by Streamlit sessions; serialize mutable predictor state.\n    with _INFERENCE_LOCK:\n        results = model.predict(\n            source=frame, conf=conf, iou=iou, imgsz=imgsz, device=device,\n            classes=class_ids, max_det=2000, verbose=False,\n        )\n\n    detections: List[Detection] = []\n    if not results:\n        return detections\n\n    boxes = results[0].boxes\n    if boxes is None or boxes.xyxy is None:\n        return detections\n\n    xyxy = boxes.xyxy.cpu().numpy()\n    confs = boxes.conf.cpu().numpy() if boxes.conf is not None else [0.0] * len(xyxy)\n\n    frame_h, frame_w = frame.shape[:2]\n\n    for (x1, y1, x2, y2), c in zip(xyxy, confs):\n        x1i = max(0, min(int(round(x1)), frame_w - 1))\n        y1i = max(0, min(int(round(y1)), frame_h - 1))\n        x2i = max(x1i + 1, min(int(round(x2)), frame_w))\n        y2i = max(y1i + 1, min(int(round(y2)), frame_h))\n\n        display_box = None\n        if native_head_class:\n            if not is_plausible_head_box(x1i, y1i, x2i, y2i):\n                continue\n            hx1, hy1, hx2, hy2 = x1i, y1i, x2i, y2i\n            if show_body_box_estimate:\n                display_box = estimate_body_box_from_head(\n                    hx1, hy1, hx2, hy2, frame_width=frame_w, frame_height=frame_h,\n                    height_multiplier=config.HEAD_TO_BODY_HEIGHT_MULTIPLIER,\n                    width_multiplier=config.HEAD_TO_BODY_WIDTH_MULTIPLIER,\n                )\n        else:\n            # Person detection mode: x1i, y1i, x2i, y2i is the full person box\n            hx1, hy1, hx2, hy2 = estimate_head_box(\n                x1i, y1i, x2i, y2i,\n                top_fraction=config.HEAD_TOP_FRACTION,\n                width_shrink=config.HEAD_WIDTH_SHRINK,\n            )\n            if show_body_box_estimate:\n                # User wants to draw body boxes -> draw the original detected person box\n                display_box = (x1i, y1i, x2i, y2i)\n\n        detections.append(Detection(hx1, hy1, hx2, hy2, float(c), display_box=display_box))\n\n    return detections\n\n\ndef run_detection(\n    weights_path: str,\n    source_path: str,\n    conf: float = config.INFERENCE_CONF_THRESHOLD,\n    iou: float = config.INFERENCE_IOU_THRESHOLD,\n    imgsz: int = config.INFERENCE_IMGSZ,\n    device: str = config.INFERENCE_DEVICE,\n    preloaded_model: Optional[YOLO] = None,\n    classification_mode: str = "manual",\n    max_capacity: Optional[int] = None,\n    show_body_box_estimate: bool = config.SHOW_BODY_BOX_ESTIMATE,\n    roi_polygon=None,\n) -> tuple:\n    """\n    Detect heads in a single image. Returns\n    (annotated_frame, head_count, crowd_level, auto_crowd_level, auto_risk_level).\n\n    `preloaded_model` is optional: pass an already-loaded YOLO instance\n    (e.g. from detection_service\'s cached model) to skip reloading the\n    model from disk on every call. If omitted, the model is loaded fresh\n    exactly as before.\n\n    `classification_mode` ("auto" or "manual") controls which crowd level\n    is BURNED INTO the returned annotated image, not just what\'s reported\n    back to the caller -- it must match whatever the dashboard/UI is using,\n    or the image overlay and the on-screen numbers will disagree.\n    """\n    if not is_valid_weights_reference(weights_path):\n        raise FileNotFoundError(f"Weights not found: {weights_path}")\n    if not os.path.isfile(source_path):\n        raise FileNotFoundError(f"Source image not found: {source_path}")\n\n    frame = cv2.imread(source_path)\n    if frame is None:\n        raise IOError(f"Failed to read image: {source_path}")\n\n    model = preloaded_model if preloaded_model is not None else load_model(weights_path)\n    class_ids = resolve_class_ids(model.names, config.TARGET_CLASS_NAMES)\n    native_head_class = _model_has_native_head_class(model)\n\n    detections = detect_heads_in_frame(\n        model, frame, conf=conf, iou=iou, imgsz=imgsz, device=device,\n        class_ids=class_ids, native_head_class=native_head_class,\n        show_body_box_estimate=show_body_box_estimate,\n    )\n\n    from src.core.scene import in_zone, draw_scene\n    detections = in_zone(detections, frame.shape, roi_polygon)\n    head_count = len(detections)\n    crowd_level = get_crowd_level(head_count, max_capacity=max_capacity if classification_mode == "manual" else None)\n    auto_crowd_level = get_auto_crowd_level(detections, frame.shape[:2])\n    auto_risk_level = get_auto_risk_level(detections, frame.shape[:2])\n\n    # This is the level actually burned into the returned image -- must\n    # match the dashboard\'s chosen mode (see docstring above).\n    display_crowd_level = auto_crowd_level if classification_mode == "auto" else crowd_level\n\n    annotated = draw_detections(frame.copy(), detections)\n    annotated = draw_overlay(annotated, head_count, display_crowd_level)\n    annotated = draw_scene(annotated, roi_polygon)\n\n    return annotated, head_count, crowd_level, auto_crowd_level, auto_risk_level\n\n\ndef main() -> None:\n    args = parse_args()\n    ensure_dir(args.save_dir)\n\n    try:\n        annotated, head_count, crowd_level, auto_crowd_level, auto_risk_level = run_detection(\n            weights_path=args.weights,\n            source_path=args.source,\n            conf=args.conf,\n            iou=args.iou,\n            imgsz=args.imgsz,\n            device=args.device,\n        )\n    except (FileNotFoundError, IOError) as exc:\n        logger.error(str(exc))\n        return\n    except Exception as exc:  # noqa: BLE001\n        logger.exception("Unexpected error during detection: %s", exc)\n        return\n\n    filename = os.path.splitext(os.path.basename(args.source))[0]\n    out_path = os.path.join(args.save_dir, f"{filename}_detected.jpg")\n    cv2.imwrite(out_path, annotated)\n\n    logger.info("Head Count: %d", head_count)\n    logger.info("Crowd Level: %s", crowd_level)\n    logger.info("Auto (density-based) Crowd Level: %s | Risk: %s", auto_crowd_level, auto_risk_level)\n    logger.info("Saved annotated image to: %s", out_path)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/core/encoding.py': '"""Direct H.264 compression without large temporary video files."""\nfrom __future__ import annotations\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport tempfile\n\n\ndef ffmpeg_executable() -> str:\n    executable = shutil.which(\'ffmpeg\')\n    if executable:\n        return executable\n    try:\n        import imageio_ffmpeg\n        return imageio_ffmpeg.get_ffmpeg_exe()\n    except (ImportError, RuntimeError) as exc:\n        raise RuntimeError(\'Video export needs FFmpeg. Install requirements.txt or add FFmpeg to PATH.\') from exc\n\n\nclass CompactVideoWriter:\n    def __init__(self, path: str, width: int, height: int, fps: float, bitrate: int, max_bytes: int = 128 * 1024**2):\n        self.max_bytes = max_bytes\n        self.path = Path(path)\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.partial = self.path.with_name(self.path.stem + \'.partial.mp4\')\n        self.log = tempfile.TemporaryFile()\n        self.closed = False\n        self.process = subprocess.Popen([\n            ffmpeg_executable(), \'-hide_banner\', \'-loglevel\', \'error\', \'-y\',\n            \'-f\', \'rawvideo\', \'-pix_fmt\', \'bgr24\', \'-s\', f\'{width}x{height}\',\n            \'-r\', str(fps), \'-i\', \'pipe:0\', \'-an\', \'-c:v\', \'libx264\',\n            \'-preset\', \'veryfast\', \'-b:v\', str(bitrate), \'-maxrate\', str(bitrate),\n            \'-bufsize\', str(bitrate * 2), \'-pix_fmt\', \'yuv420p\',\n            \'-movflags\', \'+faststart\', str(self.partial),\n        ], stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=self.log,\n            creationflags=getattr(subprocess, \'CREATE_NO_WINDOW\', 0))\n\n    def write(self, frame):\n        try:\n            self.process.stdin.write(frame.tobytes())\n            if self.partial.exists() and self.partial.stat().st_size > self.max_bytes:\n                raise RuntimeError(\'Video output exceeded its size budget. Choose a lower resolution or shorter clip.\')\n        except (BrokenPipeError, OSError) as exc:\n            raise RuntimeError(\'Video compression failed. Check disk space and FFmpeg installation.\') from exc\n\n    def close(self, success: bool = True):\n        if self.closed:\n            return\n        self.closed = True\n        try:\n            try:\n                self.process.stdin.close()\n            except (BrokenPipeError, OSError):\n                pass\n            try:\n                code = self.process.wait(timeout=60)\n            except subprocess.TimeoutExpired:\n                self.process.kill()\n                self.process.wait()\n                raise RuntimeError(\'Video encoder timed out.\')\n            if success:\n                if code:\n                    self.log.seek(0)\n                    raise RuntimeError(\'Video compression failed: \' + self.log.read().decode(errors=\'replace\')[-1200:])\n                if self.partial.stat().st_size > self.max_bytes:\n                    raise RuntimeError(\'Video output exceeded its size budget. Choose a lower resolution or shorter clip.\')\n                os.replace(self.partial, self.path)\n        finally:\n            self.log.close()\n            self.partial.unlink(missing_ok=True)\n', 'src/core/scene.py': '"""Scene geometry, sampled capacity episodes, and experimental local flow tracking."""\nimport math\nimport cv2\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\n\ndef validate_polygon(points):\n    if points is None:\n        return [[0., 0.], [1., 0.], [1., 1.], [0., 1.]]\n    polygon = np.asarray(points, dtype=float)\n    if polygon.ndim != 2 or polygon.shape[1] != 2 or not 3 <= len(polygon) <= 12:\n        raise ValueError(\'Choose 3 to 12 corners for the counting area.\')\n    if not np.isfinite(polygon).all() or (polygon < 0).any() or (polygon > 1).any():\n        raise ValueError(\'Area points must lie inside the image.\')\n    # Convex zones make selection unambiguous and reject crossing/degenerate outlines.\n    if not cv2.isContourConvex(polygon.astype(np.float32)) or abs(cv2.contourArea(polygon.astype(np.float32))) < .005:\n        raise ValueError(\'Draw a convex area in clockwise order, without crossing edges (minimum 0.5% of image).\')\n    return polygon.tolist()\n\n\ndef validate_line(points):\n    arr = np.asarray(points, dtype=float)\n    if arr.shape != (2, 2) or not np.isfinite(arr).all() or (arr < 0).any() or (arr > 1).any() or np.linalg.norm(arr[1]-arr[0]) < .05:\n        raise ValueError(\'Set two different flow-line endpoints inside the image, at least 5% apart.\')\n    return arr.tolist()\n\n\ndef in_zone(detections, shape, polygon):\n    h, w = shape[:2]\n    contour = np.asarray(validate_polygon(polygon),np.float32) * [w,h]\n    return [d for d in detections if cv2.pointPolygonTest(contour.astype(np.float32),\n            ((d.x1+d.x2)/2,(d.y1+d.y2)/2),False) >= 0]\n\n\ndef draw_scene(frame, polygon, line=None):\n    h,w=frame.shape[:2]\n    contour=(np.asarray(validate_polygon(polygon))*[w-1,h-1]).astype(np.int32)\n    cv2.polylines(frame,[contour],True,(210,195,65),2,cv2.LINE_AA)\n    if line is not None:\n        a,b=(np.asarray(validate_line(line))*[w-1,h-1]).astype(int)\n        cv2.line(frame,tuple(a),tuple(b),(70,170,255),2,cv2.LINE_AA)\n        cv2.putText(frame,\'A\',tuple(a),cv2.FONT_HERSHEY_SIMPLEX,.6,(70,170,255),2)\n        cv2.putText(frame,\'B\',tuple(b),cv2.FONT_HERSHEY_SIMPLEX,.6,(70,170,255),2)\n        vector=b-a\n        midpoint=(a+b)/2\n        endpoint=midpoint+np.array([-vector[1],vector[0]])/np.linalg.norm(vector)*min(w,h)*.07\n        cv2.arrowedLine(frame,tuple(midpoint.astype(int)),tuple(endpoint.astype(int)),(70,170,255),2,cv2.LINE_AA,tipLength=.25)\n    return frame\n\n\ndef capacity_events(counts, timestamps, capacity, minimum_seconds=3., max_gap_seconds=1.):\n    """Confirmed sampled episodes; never extrapolate beyond the final observed sample."""\n    if len(counts)!=len(timestamps) or capacity <= 0 or minimum_seconds < 0:\n        raise ValueError(\'Invalid event series or capacity.\')\n    events=[]; start=None; end=None; peak=0; peak_time=0\n    def finish():\n        if start is not None and end-start+1e-6 >= minimum_seconds:\n            events.append(dict(start_seconds=start,end_seconds=end,duration_seconds=end-start,\n                               peak_count=peak,peak_seconds=peak_time,capacity=capacity))\n    previous=None\n    for count,t in zip(counts,timestamps):\n        if not math.isfinite(t) or (previous is not None and t <= previous):\n            raise ValueError(\'Event timestamps must increase.\')\n        if previous is not None and t-previous > max_gap_seconds+1e-6:\n            finish(); start=None\n        if count > capacity:\n            if start is None:\n                start=t; peak=count; peak_time=t\n            if count > peak:\n                peak=count; peak_time=t\n            end=t\n        elif start is not None:\n            finish(); start=None\n        previous=t\n    finish()\n    return events\n\n\nclass FlowTracker:\n    """Conservative centroid association, not biometric identity or unique visitors.\n\n    Matches nearby observed heads using Hungarian assignment and short-lived IDs.\n    Crossings require two observations on each side and intersection with the finite line.\n    """\n    def __init__(self, line, max_age=1., max_distance=.08):\n        self.line=np.asarray(validate_line(line),float)\n        self.max_age=max_age; self.max_distance=max_distance\n        self.tracks={}; self.next_id=1; self.forward=0; self.reverse=0; self.events=[]\n\n    def side(self,p):\n        a,b=self.line; vector=b-a\n        distance=(vector[0]*(p[1]-a[1])-vector[1]*(p[0]-a[0]))/np.linalg.norm(vector)\n        return 1 if distance>.01 else -1 if distance<-.01 else 0\n\n    def crosses_segment(self,p,q):\n        a,b=self.line; v=b-a; move=q-p\n        determinant=v[0]*move[1]-v[1]*move[0]\n        if abs(determinant)<1e-9: return False\n        delta=p-a\n        u=(delta[0]*move[1]-delta[1]*move[0])/determinant\n        return 0 <= u <= 1\n\n    def update(self,detections,shape,timestamp):\n        h,w=shape[:2]\n        points=[np.array([(d.x1+d.x2)/(2*w),(d.y1+d.y2)/(2*h)]) for d in detections]\n        self.tracks={k:v for k,v in self.tracks.items() if timestamp-v[\'time\'] <= self.max_age}\n        ids=list(self.tracks); assigned={}\n        if ids and points:\n            costs=np.array([[np.linalg.norm(self.tracks[k][\'point\']-p) for p in points] for k in ids])\n            rows,cols=linear_sum_assignment(np.where(costs<=self.max_distance,costs,1000.))\n            assigned={int(c):ids[int(r)] for r,c in zip(rows,cols) if costs[r,c]<=self.max_distance}\n        for i,(det,p) in enumerate(zip(detections,points)):\n            tid=assigned.get(i)\n            if tid is None:\n                tid=self.next_id; self.next_id+=1\n                self.tracks[tid]=dict(point=p,time=timestamp,stable_side=0,anchor=p,candidate=0,hits=0)\n            track=self.tracks[tid]; side=self.side(p)\n            if side:\n                track[\'hits\']=track[\'hits\']+1 if track[\'candidate\']==side else 1\n                track[\'candidate\']=side\n                if track[\'hits\']>=2:\n                    if track[\'stable_side\'] and side!=track[\'stable_side\'] and self.crosses_segment(track[\'anchor\'],p):\n                        direction=\'along arrow\' if side>0 else \'against arrow\'\n                        if side>0: self.forward+=1\n                        else: self.reverse+=1\n                        self.events.append(dict(time_seconds=timestamp,direction=direction,track_id=tid))\n                        if len(self.events)>10000:\n                            raise ValueError(\'Too many flow crossings for one clip. Use a shorter recording.\')\n                    track[\'stable_side\']=side; track[\'anchor\']=p\n            track.update(point=p,time=timestamp)\n            det.track_id=tid\n        return detections\n', 'src/core/tracker.py': '\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import List, Optional\n\nimport numpy as np\nfrom ultralytics import YOLO\n\nfrom src.utils.helpers import Detection, resolve_class_ids, setup_logger\n\nlogger = setup_logger(__name__)\n\n# Path to the ByteTrack config shipped with Ultralytics.\n# Ultralytics resolves this by name automatically, no need for an absolute path.\nBYTETRACK_CONFIG = "bytetrack.yaml"\n\n\nclass HeadTracker:\n    """\n    Wraps a YOLO11 model + ByteTrack to detect and track heads across\n    a video stream. Call `reset()` between independent videos/sessions\n    so track IDs don\'t leak across unrelated streams.\n    """\n\n    def __init__(\n        self,\n        weights_path: str,\n        conf_threshold: float = 0.35,\n        iou_threshold: float = 0.5,\n        imgsz: int = 640,\n        device: str = "cpu",\n        target_class_names: tuple = ("head",),\n    ) -> None:\n        if not os.path.isfile(weights_path):\n            raise FileNotFoundError(f"YOLO weights not found at: {weights_path}")\n\n        logger.info("Loading YOLO model from %s", weights_path)\n        self.model = YOLO(weights_path)\n        self.conf_threshold = conf_threshold\n        self.iou_threshold = iou_threshold\n        self.imgsz = imgsz\n        self.device = device\n        self.class_ids = resolve_class_ids(self.model.names, target_class_names)\n        self._started = False\n\n    def reset(self) -> None:\n        """Reset internal ByteTrack state before processing a new video."""\n        predictor = getattr(self.model, "predictor", None)\n        for tracker in getattr(predictor, "trackers", ()):\n            tracker.reset()\n        self._started = False\n\n    def update(self, frame: np.ndarray) -> List[Detection]:\n        """\n        Run detection + tracking on a single frame.\n        Returns a list of Detection objects (head boxes with track IDs).\n        """\n        try:\n            results = self.model.track(\n                source=frame,\n                conf=self.conf_threshold,\n                iou=self.iou_threshold,\n                imgsz=self.imgsz,\n                device=self.device,\n                classes=self.class_ids,\n                tracker=BYTETRACK_CONFIG,\n                persist=True,\n                verbose=False,\n            )\n        except Exception as exc:  # noqa: BLE001\n            raise RuntimeError("Tracking inference failed; crowd count is unknown.") from exc\n\n        self._started = True\n        return self._parse_results(results)\n\n    @staticmethod\n    def _parse_results(results) -> List[Detection]:\n        detections: List[Detection] = []\n        if not results:\n            return detections\n\n        result = results[0]\n        boxes = result.boxes\n        if boxes is None or boxes.xyxy is None:\n            return detections\n\n        xyxy = boxes.xyxy.cpu().numpy()\n        confs = boxes.conf.cpu().numpy() if boxes.conf is not None else np.zeros(len(xyxy))\n        ids = boxes.id.cpu().numpy() if boxes.id is not None else [None] * len(xyxy)\n\n        for (x1, y1, x2, y2), conf, tid in zip(xyxy, confs, ids):\n            detections.append(\n                Detection(\n                    x1=int(round(x1)),\n                    y1=int(round(y1)),\n                    x2=int(round(x2)),\n                    y2=int(round(y2)),\n                    confidence=float(conf),\n                    track_id=int(tid) if tid is not None else None,\n                )\n            )\n        return detections\n', 'src/core/video_processor.py': '"""Sample video on its source timeline and export compact H.264 directly."""\nfrom __future__ import annotations\nfrom dataclasses import dataclass, field\nimport math\nimport os\nimport time\nfrom typing import Optional, Callable\nimport cv2\nimport config\nfrom src.core.encoding import CompactVideoWriter\nfrom src.core.detector import detect_heads_in_frame, load_model, _model_has_native_head_class\nfrom src.utils.helpers import draw_detections, draw_overlay, get_auto_crowd_level, get_crowd_level, resolve_class_ids\nfrom src.core.scene import in_zone, draw_scene, validate_polygon, FlowTracker, capacity_events\n\n\nclass AnalysisCancelled(RuntimeError):\n    pass\n\n\n@dataclass\nclass VideoProcessingResult:\n    output_path: Optional[str]\n    total_frames: int\n    processed_frames: int\n    max_head_count: int\n    avg_head_count: float\n    final_head_count: int\n    final_crowd_level: str\n    processing_time_seconds: float\n    fps_processed: float\n    per_frame_counts: list[int] = field(default_factory=list)\n    final_auto_crowd_level: str = \'Low Crowd\'\n    final_auto_risk_level: str = \'Safe\'\n    per_frame_auto_crowd_levels: list[str] = field(default_factory=list)\n    source_fps: float = 0.0\n    sample_fps: float = 0.0\n    timestamps: list[float] = field(default_factory=list)\n    peak_crowd_level: str = \'Low Crowd\'\n    source_duration: float = 0.0\n    input_bytes: int = 0\n    output_bytes: int = 0\n    output_width: int = 0\n    output_height: int = 0\n    warnings: list[str] = field(default_factory=list)\n    events: list[dict] = field(default_factory=list)\n    crossings: list[dict] = field(default_factory=list)\n    forward_crossings: int = 0\n    reverse_crossings: int = 0\n    timestamp_basis: str = \'decoder presentation timestamps\'\n\n\ndef process_video(\n    weights_path: str, source_path: str,\n    conf_threshold: float = config.INFERENCE_CONF_THRESHOLD,\n    iou_threshold: float = config.INFERENCE_IOU_THRESHOLD,\n    imgsz: int = config.INFERENCE_IMGSZ, device: str = \'cpu\',\n    output_path: Optional[str] = None, frame_callback: Optional[Callable] = None,\n    preloaded_model=None, classification_mode: str = \'manual\',\n    max_capacity: Optional[int] = None, show_body_box_estimate: bool = False,\n    analysis_fps: float = 5.0, max_width: int = 960,\n    roi_polygon=None, flow_line=None, event_seconds: float = 3.,\n    cancel_check=None, max_duration_seconds: float = 600., max_samples: int = 9000,\n) -> VideoProcessingResult:\n    if classification_mode not in (\'manual\', \'auto\'):\n        raise ValueError(\'Classification mode must be manual or auto.\')\n    if not math.isfinite(analysis_fps) or not 0 < analysis_fps <= 30:\n        raise ValueError(\'Analysis rate must be between 0 and 30 FPS.\')\n    if max_width < 64:\n        raise ValueError(\'Output width must be at least 64 pixels.\')\n    roi_polygon = validate_polygon(roi_polygon)\n    if flow_line is not None and analysis_fps < 5:\n        raise ValueError(\'Flow estimation requires at least 5 samples per second.\')\n    tracker = FlowTracker(flow_line) if flow_line is not None else None\n    if output_path and os.path.abspath(output_path) == os.path.abspath(source_path):\n        raise ValueError(\'Input and output paths must be different.\')\n    if not os.path.isfile(source_path):\n        raise FileNotFoundError(\'Source video not found.\')\n    cap = cv2.VideoCapture(source_path)\n    writer = None\n    success = False\n    started = time.perf_counter()\n    counts, levels, timestamps = [], [], []\n    try:\n        if not cap.isOpened():\n            raise IOError(\'Cannot decode this video. Try an H.264 MP4 file.\')\n        fps = cap.get(cv2.CAP_PROP_FPS)\n        if not math.isfinite(fps) or fps <= 0:\n            raise ValueError(\'The video has no valid frame rate; re-export it before analysis.\')\n        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n        width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n        if min(width, height) <= 0:\n            raise ValueError(\'The video has invalid dimensions.\')\n        sample_fps = min(fps, analysis_fps)\n        scale = min(1, max_width / width)\n        out_w = max(2, int(width * scale) // 2 * 2)\n        out_h = max(2, int(height * scale) // 2 * 2)\n        duration = total / fps if total else 0\n        if duration > max_duration_seconds:\n            raise ValueError(f\'Clip exceeds the {max_duration_seconds/60:.0f}-minute limit. Trim it before analysis.\')\n        size = os.path.getsize(source_path)\n        # Bound bitrate by source average and 2 Mbps. Never write raw/lossless intermediates.\n        bitrate = min(2_000_000, max(64_000, int(size * 8 / duration))) if duration else 1_000_000\n        model = preloaded_model if preloaded_model is not None else load_model(weights_path)\n        ids = resolve_class_ids(model.names, config.TARGET_CLASS_NAMES)\n        native = _model_has_native_head_class(model)\n        if output_path:\n            writer = CompactVideoWriter(output_path, out_w, out_h, sample_fps, bitrate, max_bytes=min(128 * 1024**2, int(size * 1.5) + 2 * 1024**2))\n        frame_idx = 0\n        next_sample = 0.\n        previous_time = -1.\n        first_pts = None\n        last_annotated = None\n        written_frames = 0\n        while True:\n            if cancel_check and cancel_check():\n                raise AnalysisCancelled(\'Analysis cancelled. Partial video removed.\')\n            if time.perf_counter()-started > 1800:\n                raise RuntimeError(\'Analysis exceeded 30 minutes. Use a shorter clip or fewer samples.\')\n            ret = cap.grab()\n            if not ret:\n                break\n            index = frame_idx\n            frame_idx += 1\n            pts = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.\n            if first_pts is None:\n                first_pts = pts\n            timestamp = pts-first_pts\n            if not math.isfinite(timestamp) or (index > 0 and timestamp <= previous_time):\n                raise ValueError(\'Decoder timestamps are missing or non-monotonic. Re-export as H.264 MP4 before analysis.\')\n            previous_time = timestamp\n            if timestamp > max_duration_seconds:\n                raise ValueError(\'Decoded clip exceeds the duration limit.\')\n            if timestamp + 1e-6 < next_sample:\n                continue\n            if len(counts) >= max_samples:\n                raise ValueError(\'Sample limit reached. Trim the clip or lower the analysis rate.\')\n            ret, frame = cap.retrieve()\n            if not ret or frame is None:\n                raise IOError(f\'Cannot decode video frame {index + 1}.\')\n            try:\n                detections = detect_heads_in_frame(\n                    model, frame, conf=conf_threshold, iou=iou_threshold,\n                    imgsz=imgsz, device=device, class_ids=ids, native_head_class=native,\n                    show_body_box_estimate=show_body_box_estimate,\n                )\n            except Exception as exc:\n                raise RuntimeError(f\'Detection failed at {timestamp:.2f}s. No valid result was saved.\') from exc\n            detections = in_zone(detections, frame.shape, roi_polygon)\n            if tracker:\n                if len(detections)>500:\n                    raise ValueError(\'Flow estimation supports at most 500 heads per sample; choose a smaller area or disable flow.\')\n                detections = tracker.update(detections, frame.shape, timestamp)\n            count = len(detections)\n            auto = get_auto_crowd_level(detections, frame.shape[:2])\n            level = auto if classification_mode == \'auto\' else get_crowd_level(count, max_capacity=max_capacity)\n            counts.append(count)\n            levels.append(auto)\n            timestamps.append(timestamp)\n            annotated = draw_overlay(draw_detections(frame.copy(), detections), count, level)\n            annotated = draw_scene(annotated, roi_polygon, flow_line)\n            annotated = cv2.resize(annotated, (out_w, out_h), interpolation=cv2.INTER_AREA)\n            if writer:\n                # Preserve gaps on the constant-rate output timeline without inventing observations.\n                target_frame = int(math.floor(timestamp*sample_fps+1e-6))\n                while last_annotated is not None and written_frames < target_frame:\n                    writer.write(last_annotated)\n                    written_frames += 1\n                writer.write(annotated)\n                written_frames += 1\n            last_annotated = annotated\n            next_sample = (math.floor(timestamp*sample_fps+1e-6)+1)/sample_fps\n            if frame_callback:\n                frame_callback(frame_idx, total, annotated, count, level, auto)\n        if not counts:\n            raise IOError(\'No readable frames were found. No analysis was saved.\')\n        if total > 0 and frame_idx < total - max(10, math.ceil(total * 0.02)):\n            raise IOError(f\'Video ended early ({frame_idx}/{total} frames). Re-export the source video.\')\n        success = True\n    finally:\n        cap.release()\n        if writer:\n            writer.close(success=success)\n    elapsed = time.perf_counter() - started\n    chosen = levels if classification_mode == \'auto\' else [get_crowd_level(c, max_capacity=max_capacity) for c in counts]\n    order = {\'Low Crowd\': 0, \'Moderate Crowd\': 1, \'High Crowd\': 2}\n    return VideoProcessingResult(\n        output_path, total, len(counts), max(counts), sum(counts) / len(counts), counts[-1], chosen[-1],\n        elapsed, len(counts) / elapsed if elapsed else 0, counts,\n        levels[-1], {\'Low Crowd\': \'Safe\', \'Moderate Crowd\': \'Warning\', \'High Crowd\': \'Critical\'}[levels[-1]], levels,\n        fps, sample_fps, timestamps, max(chosen, key=order.get), previous_time + 1/fps,\n        size, os.path.getsize(output_path) if output_path else 0, out_w, out_h,\n        [f"Container reported {total} frames; decoded {frame_idx}. Timeline uses decoded frames."] if total and frame_idx != total else [],\n        capacity_events(counts,timestamps,max_capacity or config.ASSUMED_MAX_CAPACITY,event_seconds,max(1.,2/sample_fps)),\n        tracker.events if tracker else [], tracker.forward if tracker else 0, tracker.reverse if tracker else 0,\n    )\n', 'src/core/__init__.py': '"""\nsrc.core package initialization\n"""\nfrom src.core.detector import load_model, detect_heads_in_frame, run_detection  # noqa: F401\nfrom src.core.video_processor import process_video, VideoProcessingResult  # noqa: F401\nfrom src.core.tracker import HeadTracker  # noqa: F401\n', 'src/services/detection_service.py': '"""\ndetection_service.py\n\nThin service layer sitting between the Streamlit pages and the existing\ndetection core (detect.py / video_detect.py). Centralizes model caching\n(so the YOLO model is loaded once and shared by every page in the same\nsession, instead of every page reloading it) and common validation, so\neach page file stays focused on UI code only.\n\nThis module intentionally does NOT change any detection logic in\ndetect.py / video_detect.py / utils.py -- it only wraps them.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import Callable, Optional\n\nimport streamlit as st\nfrom ultralytics import YOLO\n\nimport config\nfrom src.core.detector import run_detection\nfrom src.utils.helpers import is_valid_weights_reference, setup_logger\nfrom src.core.video_processor import (\n    VideoProcessingResult,\n    process_video,\n)\n\nlogger = setup_logger(__name__)\n\n\nclass WeightsNotFoundError(FileNotFoundError):\n    """Raised when the configured model weights path doesn\'t resolve to a usable model."""\n\n\n@st.cache_resource(show_spinner=False)\ndef _cached_model(weights_path: str, mtime: float) -> YOLO:\n    """\n    Cache the loaded YOLO model per (weights_path, mtime) so:\n      - the model is loaded only once per Streamlit session, not once per\n        page / per click, and\n      - if the user retrains and swaps best.pt, the new file\'s mtime\n        invalidates the cache automatically instead of serving a stale model.\n    """\n    return YOLO(weights_path)\n\n\ndef get_model(weights_path: str) -> YOLO:\n    """Return a cached YOLO model instance for the given weights path."""\n    if not is_valid_weights_reference(weights_path):\n        raise WeightsNotFoundError(\n            f"Model weights not found at \'{weights_path}\'. Train the model first (train.py) "\n            f"or update the path in Settings."\n        )\n    mtime = os.path.getmtime(weights_path) if os.path.isfile(weights_path) else 0.0\n    return _cached_model(weights_path, mtime)\n\n\ndef validate_weights(weights_path: str) -> Optional[str]:\n    """Return a user-friendly error message if weights are invalid, else None."""\n    if not is_valid_weights_reference(weights_path):\n        return (\n            f"Model weights not found at \'{weights_path}\'. Train the model first (train.py) "\n            f"or update the path in Settings."\n        )\n    return None\n\n\ndef run_image_detection(\n    weights_path: str, source_path: str, conf: float, iou: float, imgsz: int,\n    classification_mode: str = "manual", max_capacity: Optional[int] = None,\n    show_body_box_estimate: bool = config.SHOW_BODY_BOX_ESTIMATE,\n):\n    """\n    Run detection on a single image. Delegates to detect.run_detection, but\n    warms the shared model cache first so subsequent calls in the same\n    session are fast.\n    """\n    model = get_model(weights_path)  # warm/reuse cache; raises WeightsNotFoundError early\n    return run_detection(\n        weights_path=weights_path, source_path=source_path,\n        conf=conf, iou=iou, imgsz=imgsz, device="cpu",\n        preloaded_model=model,\n        classification_mode=classification_mode, max_capacity=max_capacity,\n        show_body_box_estimate=show_body_box_estimate,\n    )  # -> (annotated, head_count, crowd_level, auto_crowd_level, auto_risk_level)\n\n\ndef run_video_detection(\n    weights_path: str,\n    source_path: str,\n    conf: float,\n    iou: float,\n    imgsz: int,\n    output_path: str,\n    frame_callback: Optional[Callable] = None,\n    classification_mode: str = "manual",\n    max_capacity: Optional[int] = None,\n    show_body_box_estimate: bool = config.SHOW_BODY_BOX_ESTIMATE,\n    analysis_fps: float = 5.0,\n    max_width: int = 960,\n) -> VideoProcessingResult:\n    """Run frame-by-frame detection on a video. Thin pass-through to video_detect.process_video."""\n    model = get_model(weights_path)\n    return process_video(\n        weights_path=weights_path, source_path=source_path,\n        conf_threshold=conf, iou_threshold=iou, imgsz=imgsz, device="cpu",\n        output_path=output_path, frame_callback=frame_callback,\n        analysis_fps=analysis_fps, max_width=max_width,\n        preloaded_model=model,\n        classification_mode=classification_mode, max_capacity=max_capacity,\n        show_body_box_estimate=show_body_box_estimate,\n    )\n', 'src/services/exports.py': '"""CSV export uses stored source timestamps and the run\'s original classification."""\nimport csv\nimport io\n\ndef csv_bytes(record):\n    buffer=io.StringIO(newline=\'\')\n    writer=csv.writer(buffer)\n    writer.writerow([\'sample\',\'source_time_seconds\',\'count\',\'crowd_level\'])\n    counts=record[\'per_frame_counts\'];times=record[\'timestamps\'];levels=record[\'per_frame_levels\']\n    if not len(counts)==len(times)==len(levels):\n        raise ValueError(\'Frame data and source timestamps must have matching lengths.\')\n    writer.writerows(zip(range(1,len(counts)+1),times,counts,levels))\n    return buffer.getvalue().encode(\'utf-8\')\n', 'src/services/history_manager.py': '"""\nhistory_manager.py\n\nLightweight JSON-backed storage for past analysis runs (image or video),\nused by the History page and by the Reports page (to re-generate a report\nfor a previous run) and the Analytics page (to show the latest run).\n\nDeliberately dependency-free (stdlib json only) so it works the same\nwhether the caller is app.py or any Streamlit page.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport uuid\nimport threading\nfrom src.utils.privacy import redact_sensitive\nfrom src.services.storage import atomic_json\n_HISTORY_LOCK = threading.RLock()\nfrom datetime import datetime\nfrom typing import Any, Dict, List, Optional\n\nimport config\nfrom src.utils.helpers import ensure_dir, setup_logger\n\nlogger = setup_logger(__name__)\n\n\ndef _load_all() -> List[Dict[str, Any]]:\n    ensure_dir(config.HISTORY_DIR)\n    if not os.path.isfile(config.HISTORY_FILE):\n        return []\n    try:\n        with open(config.HISTORY_FILE, "r", encoding="utf-8") as f:\n            data = json.load(f)\n        return redact_sensitive(data) if isinstance(data, list) else []\n    except (json.JSONDecodeError, OSError) as exc:\n        logger.warning("Could not read history file (%s) -- starting fresh: %s", config.HISTORY_FILE, exc)\n        return []\n\n\ndef _save_all(records: List[Dict[str, Any]]) -> None:\n    ensure_dir(config.HISTORY_DIR)\n    # Keep only the most recent N entries so the file doesn\'t grow forever.\n    trimmed = records[-config.MAX_HISTORY_ENTRIES:]\n    try:\n        atomic_json(config.HISTORY_FILE, trimmed)\n    except OSError as exc:\n        logger.error("Failed to write history file: %s", exc)\n\n\ndef add_record(\n    media_type: str,\n    file_name: str,\n    max_head_count: int,\n    avg_head_count: float,\n    final_head_count: int,\n    crowd_level: str,\n    risk_level: str,\n    processing_time_seconds: float,\n    frames_processed: Optional[int] = None,\n    fps_processed: Optional[float] = None,\n    annotated_path: Optional[str] = None,\n    per_frame_counts: Optional[List[int]] = None,\n    settings: Optional[Dict[str, Any]] = None,\n) -> Dict[str, Any]:\n    """Append a new analysis record and persist it. Returns the saved record."""\n    record = {\n        "id": uuid.uuid4().hex[:12],\n        "timestamp": datetime.now().isoformat(timespec="seconds"),\n        "media_type": media_type,  # "image" or "video"\n        "file_name": file_name,\n        "max_head_count": max_head_count,\n        "avg_head_count": round(avg_head_count, 2),\n        "final_head_count": final_head_count,\n        "crowd_level": crowd_level,\n        "risk_level": risk_level,\n        "processing_time_seconds": round(processing_time_seconds, 2),\n        "frames_processed": frames_processed,\n        "fps_processed": round(fps_processed, 2) if fps_processed else None,\n        "annotated_path": annotated_path,\n        "per_frame_counts": per_frame_counts or [],\n        "settings": settings or {},\n    }\n    record = redact_sensitive(record)\n    with _HISTORY_LOCK:\n        records = _load_all()\n        records.append(record)\n        _save_all(records)\n    logger.info("History record saved: %s", record["id"])\n    return record\n\n\ndef get_all_records() -> List[Dict[str, Any]]:\n    """Return all history records, most recent first."""\n    return list(reversed(_load_all()))\n\n\ndef get_record_by_id(record_id: str) -> Optional[Dict[str, Any]]:\n    for record in _load_all():\n        if record.get("id") == record_id:\n            return record\n    return None\n\n\ndef delete_record(record_id: str) -> bool:\n    records = _load_all()\n    filtered = [r for r in records if r.get("id") != record_id]\n    if len(filtered) == len(records):\n        return False\n    _save_all(filtered)\n    return True\n\n\ndef clear_history() -> None:\n    _save_all([])\n', 'src/services/jobs.py': '"""One cancellable local process at a time; persisted state survives page reloads."""\nfrom pathlib import Path\nimport json\nimport subprocess\nimport sys\nimport os\nimport time\nimport uuid\nfrom filelock import FileLock, Timeout\nimport config\nfrom src.services.storage import atomic_json, check_storage_budget\n\nACTIVE_STATES=(\'queued\',\'running\',\'cancelling\')\n\ndef read_status(run_dir):\n    path=Path(run_dir)/\'status.json\'\n    if not path.exists(): return {}\n    try:\n        status=json.loads(path.read_text(encoding=\'utf-8\'))\n    except (ValueError,OSError): return {}\n    # An interrupted OS process must not leave a permanent running banner.\n    if status.get(\'state\') in ACTIVE_STATES and time.time()-status.get(\'updated\',0)>120:\n        lock=FileLock(str(Path(config.OUTPUTS_ROOT)/\'.analysis.lock\'))\n        try:\n            with lock.acquire(timeout=0):\n                status.update(state=\'failed\',error=\'The worker stopped unexpectedly. Run the analysis again.\')\n                atomic_json(path,status)\n        except Timeout: pass\n    return status\n\ndef start_job(run_dir, payload):\n    root=Path(config.OUTPUTS_ROOT); root.mkdir(parents=True,exist_ok=True)\n    run_dir=Path(run_dir); run_dir.mkdir(parents=True,exist_ok=True)\n    with FileLock(str(root/\'.dispatch.lock\'),timeout=5):\n        # This lock protects checks and launches across browser sessions.\n        for p in (root/\'sessions\').glob(\'*/*/status.json\'):\n            if read_status(p.parent).get(\'state\') in ACTIVE_STATES:\n                raise ValueError(\'Another analysis is running. Open its saved session or wait for it to finish.\')\n        check_storage_budget(root, 150*1024**2)\n        (run_dir/\'cancel\').unlink(missing_ok=True)\n        atomic_json(run_dir/\'job.json\',payload)\n        atomic_json(run_dir/\'status.json\',dict(state=\'queued\',updated=time.time(),progress=0.,message=\'Starting analysis\'))\n        log=open(run_dir/\'worker.log\',\'w\',encoding=\'utf-8\')\n        try:\n            proc=subprocess.Popen([sys.executable,\'-B\',\'-m\',\'src.services.worker\',str(run_dir.resolve())],\n                                  cwd=config.PROJECT_ROOT,stdin=subprocess.DEVNULL,stdout=log,stderr=log,\n                                  creationflags=getattr(subprocess,\'CREATE_NO_WINDOW\',0))\n        except Exception:\n            atomic_json(run_dir/\'status.json\',dict(state=\'failed\',updated=time.time(),error=\'Could not start worker.\'))\n            raise\n        finally: log.close()\n    return proc.pid\n\ndef cancel_job(run_dir):\n    (Path(run_dir)/\'cancel\').touch()\n\n', 'src/services/report_generator.py': '"""A PDF summary tied to the model and settings of one completed analysis."""\nfrom datetime import datetime\nfrom pathlib import Path\nfrom html import escape\nfrom reportlab.lib import colors\nfrom reportlab.lib.pagesizes import A4\nfrom reportlab.lib.styles import getSampleStyleSheet\nfrom reportlab.lib.units import cm\nfrom reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image\nimport config\nfrom src.utils.privacy import redact_sensitive\n\n\ndef generate_pdf_report(record, output_path=None, chart_image_path=None):\n    record = redact_sensitive(record)\n    path = Path(output_path) if output_path else Path(config.REPORTS_DIR)/(\'report_\' + str(record.get(\'id\',\'analysis\')) + \'.pdf\')\n    path.parent.mkdir(parents=True, exist_ok=True)\n    styles = getSampleStyleSheet()\n    def para(value, style=\'BodyText\'):\n        return Paragraph(escape(str(value)), styles[style])\n    story = [para(\'CrowdVision | Analysis report\', \'Title\'),\n             para(\'Generated \' + datetime.now().strftime(\'%Y-%m-%d %H:%M:%S\')), Spacer(1, .5*cm)]\n    rows = [\n        [\'Input\', record.get(\'file_name\',\'Unknown\')],\n        [\'Media\', record.get(\'media_type\',\'Unknown\')],\n        [\'Actual model\', record.get(\'model_name\',\'Not recorded in this historical run\')],\n        [\'Model classes\', record.get(\'model_classes\',\'Not recorded\')],\n        [\'Peak visible count\', record.get(\'max_head_count\',0)],\n        [\'Average visible count\', f"{record.get(\'avg_head_count\',0):.2f}"],\n        [\'Final visible count\', record.get(\'final_head_count\',0)],\n        [\'Peak crowd level\', record.get(\'peak_crowd_level\',\'Not recorded\')],\n        [\'Final crowd level\', record.get(\'final_crowd_level\',record.get(\'crowd_level\',\'Not recorded\'))],\n        [\'Analyzed samples\', record.get(\'processed_frames\',record.get(\'frames_processed\',\'N/A\'))],\n        [\'Processing time\', f"{record.get(\'processing_time_seconds\',0):.2f} s"],\n    ]\n    settings = record.get(\'settings\',{})\n    rows += [[\'Location\',settings.get(\'location\',\'Not recorded\')],\n             [\'Analysis completed (UTC)\',record.get(\'created_at\',\'Not recorded\')],\n             [\'Counting area\',str(settings.get(\'roi_polygon\',\'Full frame / historical run\'))],\n             [\'Sustained capacity events\',len(record.get(\'events\',[]))],\n             [\'Event duration rule\',str(settings.get(\'event_seconds\',\'Not recorded\'))+\' seconds\'],\n             [\'Along-arrow crossings\',record.get(\'forward_crossings\',0) if settings.get(\'flow_line\') else \'Disabled\'],\n             [\'Against-arrow crossings\',record.get(\'reverse_crossings\',0) if settings.get(\'flow_line\') else \'Disabled\']]\n    rows += [[\'Classification\', settings.get(\'classification_mode\',\'Not recorded\')],\n             [\'Configured capacity\', settings.get(\'max_capacity\',\'Not recorded\')],\n             [\'Source frame rate\', record.get(\'source_fps\',\'Not recorded\')],\n             [\'Sample frame rate\', record.get(\'sample_fps\',\'Not recorded\')]]\n    table = Table([[para(a),para(b)] for a,b in rows], colWidths=[6*cm,10*cm])\n    table.setStyle(TableStyle([(\'BACKGROUND\',(0,0),(0,-1),colors.HexColor(\'#edf2f7\')),\n                              (\'VALIGN\',(0,0),(-1,-1),\'TOP\'),(\'BOTTOMPADDING\',(0,0),(-1,-1),9),\n                              (\'TOPPADDING\',(0,0),(-1,-1),9),(\'LINEBELOW\',(0,0),(-1,-1),.3,colors.lightgrey)]))\n    story += [table, Spacer(1,.5*cm)]\n    if record.get(\'events\'):\n        story.append(para(\'Sampled capacity exceedances\',\'Heading2\'))\n        for event in record[\'events\'][:100]:\n            story.append(para(f"{event[\'start_seconds\']:.2f}s to {event[\'end_seconds\']:.2f}s: peak {event[\'peak_count\']}, observed duration {event[\'duration_seconds\']:.2f}s."))\n        if len(record[\'events\'])>100: story.append(para(\'First 100 events shown; download the events CSV for the complete list.\'))\n    if chart_image_path and Path(chart_image_path).is_file():\n        story.append(Image(chart_image_path,width=16*cm,height=7*cm))\n    story += [para(\'Interpretation\', \'Heading2\'),\n              para(\'Peak and final crowd levels describe different points in the recording. Counts are visible detections, not unique people. Video is sampled; brief events between samples can be missed.\'),\n              Spacer(1,.25*cm),\n              para(\'Occupancy levels use the configured capacity. Image coverage is an experimental box-area heuristic affected by perspective and zoom. Neither is a validated safety prediction. Accuracy must be evaluated for the target camera and scene.\')]\n    SimpleDocTemplate(str(path),pagesize=A4,leftMargin=2*cm,rightMargin=2*cm,topMargin=1.5*cm,bottomMargin=1.5*cm).build(story)\n    return str(path)\n', 'src/services/storage.py': '"""Immutable per-run artifacts and bounded, explicitly removable session storage."""\nfrom pathlib import Path\nimport json\nimport os\nimport shutil\nimport tempfile\nfrom src.utils.privacy import redact_sensitive\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, name = tempfile.mkstemp(dir=path.parent, suffix=\'.tmp\')\n    try:\n        with os.fdopen(fd, \'w\', encoding=\'utf-8\') as f:\n            json.dump(redact_sensitive(value), f, indent=2, allow_nan=False)\n        os.replace(name, path)\n    finally:\n        Path(name).unlink(missing_ok=True)\n\n\ndef storage_bytes(path):\n    root = Path(path)\n    size = 0\n    if root.exists():\n        for p in root.rglob(\'*\'):\n            try:\n                if p.is_file() and not p.is_symlink(): size += p.stat().st_size\n            except FileNotFoundError:\n                pass\n    return size\n\n\ndef session_inventory(root):\n    """Only app-owned UUID session folders; never traverse linked directories."""\n    import re\n    root=Path(root)\n    if not root.exists(): return []\n    sessions=[]\n    for directory in root.iterdir():\n        if directory.is_dir() and not directory.is_symlink() and re.fullmatch(r\'[a-f0-9]{32}\',directory.name):\n            runs=[]\n            for record in directory.glob(\'*/result.json\'):\n                if record.parent.is_symlink(): continue\n                try:\n                    data=json.loads(record.read_text(encoding=\'utf-8\'))\n                    output=Path(data[\'annotated_path\']).resolve()\n                    if output.parent==record.parent.resolve() and output.is_file():\n                        runs.append(data)\n                except (OSError,ValueError,KeyError): pass\n            sessions.append(dict(id=directory.name,bytes=storage_bytes(directory),runs=runs,modified=directory.stat().st_mtime))\n    return sorted(sessions,key=lambda x:x[\'modified\'],reverse=True)\n\n\ndef check_storage_budget(root, additional_bytes, limit=2*1024**3):\n    """Fail before writing; callers serialize writes/jobs with the local workspace lock."""\n    reserved=0\n    for path in (Path(root)/\'sessions\').glob(\'*/*/status.json\'):\n        try:\n            state=json.loads(path.read_text(encoding=\'utf-8\')).get(\'state\')\n            if state in (\'queued\',\'running\',\'cancelling\'):\n                reserved+=max(0,150*1024**2-storage_bytes(path.parent))\n        except (OSError,ValueError): pass\n    if storage_bytes(root)+reserved+additional_bytes > limit:\n        raise ValueError(\'The 2 GB workspace limit would be exceeded. Remove saved sessions in Storage before continuing.\')\n\n\ndef clear_session(root, session_id):\n    import re\n    root = Path(root).resolve()\n    original=root/session_id\n    if not re.fullmatch(r\'[a-f0-9]{32}\',session_id) or original.is_symlink() or original.is_junction():\n        raise ValueError(\'Invalid session path.\')\n    target = (root / session_id).resolve()\n    if target.parent != root or not session_id or target == root:\n        raise ValueError(\'Invalid session path.\')\n    if target.exists():\n        shutil.rmtree(target)\n\n\ndef retain_run(session_dir, signature):\n    """Keep one completed run per session. Only remove our signature-named run folders."""\n    import re\n    root = Path(session_dir).resolve()\n    for path in root.iterdir():\n        if path.is_dir() and not path.is_symlink() and re.fullmatch(r"[0-9a-f]{24}", path.name) and path.name != signature:\n            if path.resolve().parent == root:\n                shutil.rmtree(path)\n\n\nLEGACY_FOLDERS = (\'streamlit_videos\', \'streamlit_images\', \'rtsp_recordings\', \'_uploads\', \'csv\', \'reports\')\n\ndef clear_legacy_exports(outputs_root, folders):\n    """Delete only explicitly selected direct legacy exports, never source media elsewhere."""\n    root = Path(outputs_root).resolve()\n    if any(folder not in LEGACY_FOLDERS for folder in folders):\n        raise ValueError(\'Select a supported generated-output folder.\')\n    removed = 0\n    for folder in folders:\n        original = root / folder\n        target = original.resolve()\n        if original.is_symlink() or target.parent != root:\n            raise ValueError(\'Unsafe cleanup path; nothing deleted in this folder.\')\n        if target.exists():\n            for path in target.iterdir():\n                if path.is_file() and not path.is_symlink():\n                    removed += path.stat().st_size\n                    path.unlink()\n    return removed\n', 'src/services/worker.py': '"""Background analysis process. No Streamlit calls or shared model predictor state."""\nfrom pathlib import Path\nfrom dataclasses import asdict\nfrom datetime import datetime, timezone\nimport json\nimport sys\nimport time\nimport cv2\nimport torch\nfrom filelock import FileLock\nimport config\nfrom src.core.detector import load_model, run_detection\nfrom src.core.video_processor import process_video, AnalysisCancelled\nfrom src.services.storage import atomic_json, check_storage_budget\nfrom src.utils.helpers import get_crowd_level\n\n\ndef run(run_dir):\n    run_dir=Path(run_dir)\n    job=json.loads((run_dir/\'job.json\').read_text(encoding=\'utf-8\'))\n    settings=job[\'settings\']; start=time.perf_counter(); last=[0.]\n    torch.set_num_threads(2)\n    def status(state, **values):\n        atomic_json(run_dir/\'status.json\',dict(state=state,updated=time.time(),**values))\n    def cancelled(): return (run_dir/\'cancel\').exists()\n    def progress(index,total,frame,count,level,auto):\n        now=time.perf_counter()\n        if now-last[0]>.8:\n            last[0]=now\n            thumb=cv2.resize(frame,(640,max(2,int(frame.shape[0]*640/frame.shape[1]))))\n            cv2.imwrite(str(run_dir/\'preview.tmp.jpg\'),thumb,[cv2.IMWRITE_JPEG_QUALITY,70])\n            (run_dir/\'preview.tmp.jpg\').replace(run_dir/\'preview.jpg\')\n            status(\'running\',progress=min(index/total,.99) if total else 0.,\n                   message=f\'Frame {index:,} / {total:,} | {count} heads in area\',elapsed_seconds=now-start)\n    try:\n        with FileLock(str(Path(config.OUTPUTS_ROOT)/\'.analysis.lock\'),timeout=0):\n            status(\'running\',progress=0.,message=\'Loading detector\')\n            check_storage_budget(config.OUTPUTS_ROOT,0)\n            model=load_model(settings[\'weights_path\'])\n            common=dict(weights_path=settings[\'weights_path\'],source_path=job[\'source_path\'],\n                        imgsz=settings[\'imgsz\'],preloaded_model=model,device=\'cpu\',\n                        max_capacity=settings[\'max_capacity\'],roi_polygon=settings[\'roi_polygon\'])\n            if cancelled(): raise AnalysisCancelled(\'Analysis cancelled.\')\n            if job[\'media_type\']==\'video\':\n                output=run_dir/\'annotated.mp4\'\n                result=process_video(**common,conf_threshold=settings[\'conf\'],iou_threshold=.5,\n                                     output_path=str(output),analysis_fps=settings[\'analysis_fps\'],\n                                     max_width=settings[\'max_width\'],flow_line=settings.get(\'flow_line\'),\n                                     event_seconds=settings[\'event_seconds\'],frame_callback=progress,cancel_check=cancelled)\n                data=asdict(result)\n                data[\'per_frame_levels\']=[get_crowd_level(n,settings[\'max_capacity\']) for n in result.per_frame_counts]\n                cap=cv2.VideoCapture(str(output))\n                try:\n                    # Store a bounded number of evidence thumbnails, ordered by event severity.\n                    for i,event in enumerate(sorted(data[\'events\'],key=lambda e:e[\'peak_count\'],reverse=True)[:12]):\n                        cap.set(cv2.CAP_PROP_POS_MSEC,event[\'peak_seconds\']*1000)\n                        ok,frame=cap.read()\n                        if ok:\n                            image=run_dir/f\'event_{i:02d}.jpg\'\n                            frame=cv2.resize(frame,(480,max(2,int(frame.shape[0]*480/frame.shape[1]))))\n                            cv2.imwrite(str(image),frame,[cv2.IMWRITE_JPEG_QUALITY,75])\n                            event[\'thumbnail\']=str(image)\n                finally: cap.release()\n            else:\n                output=run_dir/\'annotated.jpg\'\n                annotated,count,level,_,_=run_detection(**common,conf=settings[\'conf\'],iou=.5)\n                if not cv2.imwrite(str(output),annotated,[cv2.IMWRITE_JPEG_QUALITY,88]):\n                    raise IOError(\'Cannot write annotated image.\')\n                data=dict(max_head_count=count,avg_head_count=float(count),final_head_count=count,\n                          final_crowd_level=level,peak_crowd_level=level,per_frame_counts=[count],\n                          per_frame_levels=[level],timestamps=[0.],processed_frames=1,\n                          input_bytes=Path(job[\'source_path\']).stat().st_size,output_bytes=output.stat().st_size,\n                          processing_time_seconds=time.perf_counter()-start,events=[],crossings=[],\n                          forward_crossings=0,reverse_crossings=0,source_duration=0,source_fps=None,sample_fps=None)\n            if cancelled():\n                output.unlink(missing_ok=True)\n                raise AnalysisCancelled(\'Analysis cancelled.\')\n            record=dict(data,signature=run_dir.name,id=run_dir.name,file_name=job[\'file_name\'],\n                        media_type=job[\'media_type\'],annotated_path=str(output),source_path=job[\'source_path\'],\n                        settings=settings,model_name=str(model.model.yaml.get(\'yaml_file\',\'YOLO\')),\n                        model_classes=\', \'.join(model.names.values()),crowd_level=data[\'peak_crowd_level\'],\n                        created_at=datetime.now(timezone.utc).isoformat())\n            atomic_json(run_dir/\'result.json\',record)\n            status(\'complete\',progress=1.,message=\'Analysis complete\')\n    except AnalysisCancelled as exc:\n        status(\'cancelled\',error=str(exc))\n    except Exception as exc:\n        status(\'failed\',error=str(exc))\n    finally:\n        (run_dir/\'preview.tmp.jpg\').unlink(missing_ok=True)\n\nif __name__==\'__main__\': run(sys.argv[1])\n', 'src/services/__init__.py': '"""\nsrc.services package initialization\n"""\nfrom src.services.detection_service import get_model, validate_weights, run_image_detection, run_video_detection  # noqa: F401\nfrom src.services.history_manager import add_record, get_all_records, get_record_by_id  # noqa: F401\nfrom src.services.report_generator import generate_pdf_report  # noqa: F401\n', 'src/ui/app_state.py': '"""\napp_state.py\n\nShared Streamlit session-state initialization and CSS theming for the\nsingle-page CrowdVision AI app. Premium dark glassmorphism theme.\n"""\n\nfrom __future__ import annotations\n\nimport streamlit as st\n\nimport config\n\nSETTINGS_KEY = "app_settings"\n\n\ndef init_app_state() -> None:\n    """Initialize all session-state defaults exactly once."""\n    if SETTINGS_KEY not in st.session_state:\n        st.session_state[SETTINGS_KEY] = {\n            "weights_path": config.BEST_WEIGHTS_PATH,\n            "conf": config.INFERENCE_CONF_THRESHOLD,\n            "iou": config.INFERENCE_IOU_THRESHOLD,\n            "imgsz": config.INFERENCE_IMGSZ,\n            "max_capacity": config.ASSUMED_MAX_CAPACITY,\n            "classification_mode": "auto",\n            "show_body_box_estimate": config.SHOW_BODY_BOX_ESTIMATE,\n        }\n\n    defaults = {\n        "annotated_path": None,\n        "head_count": None,\n        "crowd_level": None,\n        "risk_level": None,\n        "processing_time": None,\n        "error": None,\n        "last_image_record": None,\n        "last_uploaded_name": None,\n        "video_output_path": None,\n        "video_final_head_count": None,\n        "video_final_crowd_level": None,\n        "video_final_risk_level": None,\n        "video_max_head_count": None,\n        "video_avg_head_count": None,\n        "video_processing_time": None,\n        "video_per_frame_counts": None,\n        "video_per_frame_auto_levels": None,\n        "video_frames_processed": None,\n        "video_fps_processed": None,\n        "video_error": None,\n        "last_video_record": None,\n        "last_csv_path": None,\n        "last_heatmap_path": None,\n    }\n    for key, val in defaults.items():\n        if key not in st.session_state:\n            st.session_state[key] = val\n\n\ndef get_settings() -> dict:\n    init_app_state()\n    return st.session_state[SETTINGS_KEY]\n\n\ndef update_settings(**kwargs) -> None:\n    init_app_state()\n    st.session_state[SETTINGS_KEY].update(kwargs)\n\n\ndef configure_page() -> None:\n    """st.set_page_config + CSS injection. Called once at the top of app.py."""\n    st.set_page_config(\n        page_title=f"CrowdVision AI · {config.PROJECT_DISPLAY_NAME}",\n        page_icon="🧍",\n        layout="wide",\n        initial_sidebar_state="collapsed",\n    )\n    init_app_state()\n    _inject_css()\n\n\ndef _inject_css() -> None:\n    st.markdown(\n        """\n        <style>\n        @import url(\'https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Inter:wght@400;500;600;700&display=swap\');\n\n        html, body, [class*="css"] {\n            font-family: \'Plus Jakarta Sans\', \'Inter\', sans-serif;\n            color: #F1F3F9;\n        }\n\n        :root {\n            --bg: #070A12;\n            --card-bg: rgba(255, 255, 255, 0.035);\n            --card-border: rgba(255, 255, 255, 0.08);\n            --text-primary: #F8FAFC;\n            --text-secondary: #94A3B8;\n            --accent-blue: #4F6BFF;\n            --accent-purple: #9B5CFF;\n            --accent-green: #10B981;\n            --accent-orange: #F59E0B;\n            --accent-red: #EF4444;\n        }\n\n        /* Hide Streamlit chrome */\n        #MainMenu, header[data-testid="stHeader"], footer, [data-testid="stToolbar"] {\n            visibility: hidden; height: 0; }\n        div[data-testid="stDecoration"] { display: none; }\n        section[data-testid="stSidebar"] { display: none; }\n\n        .stApp {\n            background:\n                radial-gradient(1400px 700px at 15% -15%, rgba(79,107,255,0.12), transparent 70%),\n                radial-gradient(1200px 600px at 85% 10%, rgba(155,92,255,0.12), transparent 70%),\n                radial-gradient(800px 400px at 50% 100%, rgba(16,185,129,0.05), transparent 60%),\n                var(--bg);\n            background-attachment: fixed;\n        }\n\n        .block-container {\n            padding-top: 1.2rem;\n            padding-bottom: 2rem;\n            max-width: 1400px;\n        }\n\n        /* ---- Glass card ---- */\n        .glass-card {\n            background: var(--card-bg);\n            border: 1px solid var(--card-border);\n            border-radius: 18px;\n            padding: 22px 24px;\n            backdrop-filter: blur(16px);\n            -webkit-backdrop-filter: blur(16px);\n            box-shadow: 0 8px 32px rgba(0, 0, 0, 0.35);\n            margin-bottom: 20px;\n            transition: transform 0.2s ease, box-shadow 0.2s ease, border-color 0.2s ease;\n        }\n        .glass-card:hover {\n            border-color: rgba(255, 255, 255, 0.14);\n        }\n        .card-title {\n            display: flex; align-items: center; gap: 10px;\n            font-size: 1rem; font-weight: 700; color: var(--text-primary);\n            margin-bottom: 16px; letter-spacing: -0.01em;\n        }\n\n        /* ---- Top bar ---- */\n        .topbar {\n            display: flex; align-items: center; justify-content: space-between;\n            padding: 4px 2px 18px 2px;\n        }\n        .topbar .brand { display: flex; align-items: center; gap: 14px; }\n        .topbar .logo-badge {\n            width: 44px; height: 44px; border-radius: 14px;\n            background: linear-gradient(135deg, var(--accent-blue), var(--accent-purple));\n            display: flex; align-items: center; justify-content: center;\n            font-size: 1.4rem; box-shadow: 0 6px 20px rgba(79,107,255,0.4);\n        }\n        .topbar .brand-title { font-size: 1.45rem; font-weight: 800; color: var(--text-primary); line-height: 1.1; letter-spacing: -0.02em; }\n        .topbar .brand-title span { color: var(--accent-purple); }\n        .topbar .brand-subtitle { font-size: 0.8rem; color: var(--text-secondary); margin-top: 2px; font-weight: 500; }\n\n        /* ---- Hero / info strip ---- */\n        .info-strip {\n            display: flex; align-items: center; justify-content: space-between;\n            gap: 20px; flex-wrap: wrap;\n        }\n        .info-strip .info-left { max-width: 680px; }\n        .info-strip h2 { margin: 0 0 6px 0; font-size: 1.4rem; color: var(--text-primary); font-weight: 800; letter-spacing: -0.02em; }\n        .info-strip p { margin: 0; color: var(--text-secondary); font-size: 0.92rem; line-height: 1.5; }\n        .badge-row { display: flex; gap: 10px; flex-wrap: wrap; }\n        .pill {\n            padding: 7px 16px; border-radius: 999px; font-size: 0.78rem; font-weight: 600;\n            border: 1px solid var(--card-border); white-space: nowrap; backdrop-filter: blur(8px);\n            transition: all 0.2s ease;\n        }\n        .pill-blue   { background: rgba(79,107,255,0.14);  color: #A5B4FC; border-color: rgba(79,107,255,0.25); }\n        .pill-purple { background: rgba(155,92,255,0.14);  color: #DDD6FE; border-color: rgba(155,92,255,0.25); }\n        .pill-green  { background: rgba(16,185,129,0.14);   color: #6EE7B7; border-color: rgba(16,185,129,0.25); }\n        .pill-orange { background: rgba(245,158,11,0.14);  color: #FDE68A; border-color: rgba(245,158,11,0.25); }\n\n        /* ---- Metric tiles ---- */\n        .metric-tile {\n            display: flex; align-items: center; gap: 14px;\n            background: rgba(255,255,255,0.025);\n            border: 1px solid var(--card-border);\n            border-radius: 16px; padding: 16px 18px;\n            transition: transform 0.2s ease, border-color 0.2s ease;\n        }\n        .metric-tile:hover {\n            border-color: rgba(255,255,255,0.15);\n            transform: translateY(-2px);\n        }\n        .metric-tile .icon-badge {\n            width: 42px; height: 42px; min-width: 42px; border-radius: 12px;\n            display: flex; align-items: center; justify-content: center; font-size: 1.15rem;\n            box-shadow: 0 4px 12px rgba(0,0,0,0.2);\n        }\n        .metric-tile .m-label { font-size: 0.76rem; color: var(--text-secondary); margin-bottom: 3px; font-weight: 500; }\n        .metric-tile .m-value { font-size: 1.4rem; font-weight: 800; color: var(--text-primary); line-height: 1.1; letter-spacing: -0.02em; }\n        .metric-tile .m-sub { font-size: 0.74rem; color: var(--text-secondary); margin-top: 3px; font-weight: 500; }\n\n        .risk-badge {\n            display: inline-flex; align-items: center; gap: 6px;\n            padding: 6px 14px; border-radius: 999px; font-size: 0.84rem; font-weight: 700;\n        }\n        .risk-safe     { background: rgba(16,185,129,0.16);  color: #6EE7B7; border: 1px solid rgba(16,185,129,0.3); }\n        .risk-warning  { background: rgba(245,158,11,0.16); color: #FDE68A; border: 1px solid rgba(245,158,11,0.3); }\n        .risk-critical { background: rgba(239,68,68,0.18);  color: #FCA5A5; border: 1px solid rgba(239,68,68,0.3); }\n\n        /* Buttons */\n        div.stButton > button, div.stDownloadButton > button {\n            border-radius: 12px !important;\n            font-weight: 600 !important;\n            border: 1px solid var(--card-border) !important;\n            transition: all 0.2s ease !important;\n            font-family: \'Plus Jakarta Sans\', sans-serif !important;\n            padding: 0.55rem 1.2rem !important;\n        }\n        div.stButton > button:hover, div.stDownloadButton > button:hover {\n            border-color: rgba(255, 255, 255, 0.25) !important;\n            transform: translateY(-1px);\n        }\n        div.stButton > button[kind="primary"], div.stDownloadButton > button[kind="primary"] {\n            background: linear-gradient(135deg, var(--accent-blue), var(--accent-purple)) !important;\n            border: none !important;\n            box-shadow: 0 4px 18px rgba(79,107,255,0.35) !important;\n        }\n        div.stButton > button[kind="primary"]:hover, div.stDownloadButton > button[kind="primary"]:hover {\n            box-shadow: 0 6px 24px rgba(79,107,255,0.5) !important;\n            transform: translateY(-1px);\n        }\n\n        /* File uploader */\n        [data-testid="stFileUploaderDropzone"] {\n            background: rgba(255,255,255,0.018) !important;\n            border: 1.5px dashed rgba(255,255,255,0.16) !important;\n            border-radius: 16px !important;\n            transition: all 0.2s ease !important;\n            padding: 1.5rem !important;\n        }\n        [data-testid="stFileUploaderDropzone"]:hover {\n            border-color: var(--accent-blue) !important;\n            background: rgba(79,107,255,0.04) !important;\n        }\n\n        hr { border-color: var(--card-border); }\n        </style>\n        """,\n        unsafe_allow_html=True,\n    )\n\n\ndef risk_badge_html(risk_level: str) -> str:\n    css_class = {"Safe": "risk-safe", "Warning": "risk-warning", "Critical": "risk-critical"}.get(\n        risk_level, "risk-warning"\n    )\n    icon = config.RISK_LEVEL_ICONS.get(risk_level, "⚪")\n    return f\'<span class="risk-badge {css_class}">{icon} {risk_level}</span>\'\n', 'src/ui/scene_editor.py': '"""Local, dependency-free Streamlit canvas component for scene geometry."""\nfrom pathlib import Path\nimport base64\nimport cv2\nimport streamlit.components.v1 as components\n\n_editor=components.declare_component(\'crowdvision_scene\',path=str(Path(__file__).parent/\'scene_component\'))\n\ndef scene_editor(frame, key, geometry=None):\n    height,width=frame.shape[:2]\n    scale=min(1,960/width)\n    preview=cv2.resize(frame,(int(width*scale),int(height*scale)))\n    ok,encoded=cv2.imencode(\'.jpg\',preview,[cv2.IMWRITE_JPEG_QUALITY,80])\n    if not ok: raise ValueError(\'Cannot prepare source preview.\')\n    geometry=geometry or {\'polygon\':None,\'line\':None}\n    return _editor(image=\'data:image/jpeg;base64,\'+base64.b64encode(encoded).decode(),\n                   geometry=geometry,key=key,default=geometry)\n', 'src/ui/__init__.py': '"""\nsrc.ui package initialization\n"""\nfrom src.ui.app_state import configure_page, get_settings, update_settings, risk_badge_html  # noqa: F401\n', 'src/utils/helpers.py': '"""\nutils.py\n\nShared utility functions for the Crowd Management System.\nIncludes crowd-level classification, drawing helpers, logging setup,\nand misc file / time helpers. CPU-friendly, no heavy dependencies\nbeyond OpenCV and NumPy.\n"""\n\nfrom __future__ import annotations\n\nimport logging\nimport os\nimport time\nfrom dataclasses import dataclass\nfrom typing import Iterable, Optional, Sequence, Tuple\n\nimport cv2\nimport numpy as np\n\nimport config\n\n# --------------------------------------------------------------------------- #\n# Constants (sourced from config.py -- edit values there, not here)\n# --------------------------------------------------------------------------- #\n\nBOX_COLOR: Tuple[int, int, int] = config.BOX_COLOR_BGR\nTEXT_COLOR: Tuple[int, int, int] = config.TEXT_COLOR_BGR\nBOX_THICKNESS: int = config.BOX_THICKNESS\nFONT = cv2.FONT_HERSHEY_SIMPLEX\n\nLOW_CROWD_MAX = config.LOW_CROWD_MAX\nMODERATE_CROWD_MAX = config.MODERATE_CROWD_MAX\n\nCROWD_LEVEL_COLORS = config.CROWD_LEVEL_COLORS_BGR\n\n\n# --------------------------------------------------------------------------- #\n# Logging\n# --------------------------------------------------------------------------- #\n\ndef setup_logger(name: str = "crowd_management", level: int = logging.INFO) -> logging.Logger:\n    """Create (or fetch) a configured logger with a consistent format."""\n    logger = logging.getLogger(name)\n    if logger.handlers:\n        # Already configured -> avoid duplicate handlers on repeated imports\n        return logger\n\n    logger.setLevel(level)\n    handler = logging.StreamHandler()\n    formatter = logging.Formatter(\n        fmt="[%(asctime)s] [%(levelname)s] [%(name)s] %(message)s",\n        datefmt="%Y-%m-%d %H:%M:%S",\n    )\n    handler.setFormatter(formatter)\n    logger.addHandler(handler)\n    return logger\n\n\n# --------------------------------------------------------------------------- #\n# Automatic, density-based classification\n# --------------------------------------------------------------------------- #\n#\n# Raw head COUNT can\'t tell a tightly-packed close-up from a stadium shot\n# apart -- 50 heads filling a small room is a crush, 50 heads scattered\n# across a stadium\'s upper tier (or seen from a wide/crooked overhead\n# angle) is nothing. Counting alone also has no notion of "how full does\n# this frame actually look".\n#\n# Instead, PACKING DENSITY -- what fraction of the visible frame area the\n# detected heads themselves cover -- is computed directly from the boxes\n# YOLO already returns and the frame\'s own dimensions. This scales itself\n# automatically to whatever is on screen: a distant stadium tier, a close\n# hallway, an oblique/crooked security-camera angle -- no manual "capacity"\n# number to type in and get wrong. Any Optional[int] capacity is now only\n# an *override* for someone who genuinely wants a fixed-headcount rule for\n# a specific, known space; automatic mode is the default in the app.\n\n# Fraction of total frame area covered by head boxes at each boundary.\n# Tuned so a handful of close-up faces (large boxes) and a genuine crowd\n# of small, distant boxes both land in a sensible bucket.\nDENSITY_LOW_MAX = 0.015     # <=1.5% of frame covered by heads -> Low\nDENSITY_MODERATE_MAX = 0.05  # <=5% of frame covered by heads -> Moderate\n\n\ndef compute_packing_density(detections: Sequence["Detection"], frame_shape: Tuple[int, int]) -> float:\n    """\n    Fraction (0-1) of the frame\'s area covered by detected head boxes.\n\n    `frame_shape` is (height, width) -- e.g. `frame.shape[:2]` from OpenCV.\n    Overlapping boxes are summed rather than de-duplicated, which is\n    intentional: overlap itself is a signal of crowding, not noise.\n    """\n    height, width = frame_shape[0], frame_shape[1]\n    frame_area = max(height * width, 1)\n    covered = 0\n    for det in detections:\n        w = max(det.x2 - det.x1, 0)\n        h = max(det.y2 - det.y1, 0)\n        covered += w * h\n    return covered / frame_area\n\n\ndef get_auto_crowd_level(detections: Sequence["Detection"], frame_shape: Tuple[int, int]) -> str:\n    """Classify Low/Moderate/High crowd level from packing density, not raw count."""\n    density = compute_packing_density(detections, frame_shape)\n    if density <= DENSITY_LOW_MAX:\n        return "Low Crowd"\n    if density <= DENSITY_MODERATE_MAX:\n        return "Moderate Crowd"\n    return "High Crowd"\n\n\ndef get_auto_risk_level(detections: Sequence["Detection"], frame_shape: Tuple[int, int]) -> str:\n    """Safe/Warning/Critical counterpart of get_auto_crowd_level(), same thresholds."""\n    level = get_auto_crowd_level(detections, frame_shape)\n    return {"Low Crowd": "Safe", "Moderate Crowd": "Warning", "High Crowd": "Critical"}[level]\n\n\ndef get_crowd_level(count: int, max_capacity: Optional[int] = None) -> str:\n    """\n    Classify a head-count into Low / Moderate / High crowd level.\n\n    If `max_capacity` is given, thresholds scale with it (as a % of\n    capacity) instead of using the fixed absolute defaults below. This\n    matters for venues like a stadium: a fixed "40 heads = High Crowd"\n    rule makes every wide/crooked-angle stadium shot register as\n    permanently critical, even at normal occupancy. Pass the scene\'s\n    configured capacity (e.g. from Settings) to get a level that\'s\n    meaningful for that specific venue.\n    """\n    if count < 0:\n        raise ValueError("count must be a non-negative integer")\n\n    if max_capacity and max_capacity > 0:\n        # Explicit occupancy bands; heuristic rules, not validated safety thresholds.\n        low_max = max_capacity * 0.60\n        moderate_max = max_capacity * 0.85\n        if count <= low_max:\n            return "Low Crowd"\n        if count <= moderate_max:\n            return "Moderate Crowd"\n        return "High Crowd"\n\n    if count <= LOW_CROWD_MAX:\n        return "Low Crowd"\n    if count <= MODERATE_CROWD_MAX:\n        return "Moderate Crowd"\n    return "High Crowd"\n\n\ndef get_crowd_level_color(level: str) -> Tuple[int, int, int]:\n    """Return a BGR color associated with a crowd level for overlay text."""\n    return CROWD_LEVEL_COLORS.get(level, (255, 255, 255))\n\n\ndef get_risk_level(count: int, max_capacity: Optional[int] = None) -> str:\n    """\n    Classify a head-count into a safety-oriented risk level: Safe / Warning /\n    Critical. Uses the same thresholds as get_crowd_level() (fixed by\n    default, or scaled to `max_capacity` when provided) so the two stay\n    consistent, but is exposed separately since "risk" is the framing used\n    in the Analytics / Reports pages.\n    """\n    if count < 0:\n        raise ValueError("count must be a non-negative integer")\n\n    if max_capacity and max_capacity > 0:\n        level = get_crowd_level(count, max_capacity=max_capacity)\n        return {"Low Crowd": "Safe", "Moderate Crowd": "Warning", "High Crowd": "Critical"}[level]\n\n    if count <= config.RISK_LOW_MAX:\n        return "Safe"\n    if count <= config.RISK_MODERATE_MAX:\n        return "Warning"\n    return "Critical"\n\n\ndef get_risk_level_icon(level: str) -> str:\n    """Return the emoji icon associated with a risk level."""\n    return config.RISK_LEVEL_ICONS.get(level, "⚪")\n\n\ndef get_density_percentage(count: int, max_capacity: Optional[int] = None) -> float:\n    """\n    Compute an occupancy percentage relative to an assumed maximum capacity.\n    Values above 100 are intentionally not clamped -- the UI uses that to\n    flag genuine overcrowding rather than silently capping it at 100%.\n    """\n    capacity = max_capacity or config.ASSUMED_MAX_CAPACITY\n    if capacity <= 0:\n        return 0.0\n    return round((count / capacity) * 100.0, 1)\n\n\n# --------------------------------------------------------------------------- #\n# Data container for a single detection\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass Detection:\n    x1: int\n    y1: int\n    x2: int\n    y2: int\n    confidence: float\n    track_id: Optional[int] = None\n    # Optional alternate box used ONLY for drawing (see\n    # estimate_body_box_from_head). When set, draw_detections() draws this\n    # instead of (x1, y1, x2, y2). Head-count and packing-density\n    # calculations always use the real (x1, y1, x2, y2) box, never this one.\n    display_box: Optional[Tuple[int, int, int, int]] = None\n\n\nclass BoxSmoother:\n    """\n    Exponential Moving Average (EMA) bounding box coordinate smoother across video frames.\n    Prevents visual box jittering and flickering in live streams / video playback.\n    """\n\n    def __init__(self, alpha: float = 0.45) -> None:\n        self.alpha = alpha\n        self.prev_boxes: dict[int, Tuple[float, float, float, float]] = {}\n\n    def reset(self) -> None:\n        self.prev_boxes.clear()\n\n    def smooth(self, detections: List[Detection]) -> List[Detection]:\n        if not detections:\n            self.prev_boxes.clear()\n            return detections\n\n        smoothed_detections: List[Detection] = []\n        current_ids: set[int] = set()\n\n        for idx, det in enumerate(detections):\n            # Key by track_id if available, otherwise by detection index\n            if det.track_id is None:\n                # Detection order is unstable. Never mix coordinates across people.\n                smoothed_detections.append(det)\n                continue\n            key = det.track_id\n            current_ids.add(key)\n\n            x1, y1, x2, y2 = float(det.x1), float(det.y1), float(det.x2), float(det.y2)\n\n            if key in self.prev_boxes:\n                px1, py1, px2, py2 = self.prev_boxes[key]\n                sx1 = self.alpha * x1 + (1.0 - self.alpha) * px1\n                sy1 = self.alpha * y1 + (1.0 - self.alpha) * py1\n                sx2 = self.alpha * x2 + (1.0 - self.alpha) * px2\n                sy2 = self.alpha * y2 + (1.0 - self.alpha) * py2\n            else:\n                sx1, sy1, sx2, sy2 = x1, y1, x2, y2\n\n            self.prev_boxes[key] = (sx1, sy1, sx2, sy2)\n\n            # Smooth display_box if present\n            disp = None\n            if det.display_box is not None:\n                dx1, dy1, dx2, dy2 = (float(v) for v in det.display_box)\n                disp = (\n                    int(round(dx1)),\n                    int(round(dy1)),\n                    int(round(dx2)),\n                    int(round(dy2)),\n                )\n\n            smoothed_detections.append(\n                Detection(\n                    x1=int(round(sx1)),\n                    y1=int(round(sy1)),\n                    x2=int(round(sx2)),\n                    y2=int(round(sy2)),\n                    confidence=det.confidence,\n                    track_id=det.track_id,\n                    display_box=disp,\n                )\n            )\n\n        # Evict missing track IDs\n        stale_keys = [k for k in self.prev_boxes if k not in current_ids]\n        for k in stale_keys:\n            del self.prev_boxes[k]\n\n        return smoothed_detections\n\n\n# --------------------------------------------------------------------------- #\n# Drawing helpers\n# --------------------------------------------------------------------------- #\n\ndef draw_detections(frame: np.ndarray, detections: Iterable[Detection]) -> np.ndarray:\n    """Draw blue bounding boxes + track-id labels for every detected head."""\n    h, w = frame.shape[:2]\n    for det in detections:\n        raw_box = det.display_box if det.display_box is not None else (det.x1, det.y1, det.x2, det.y2)\n        bx1 = max(0, min(int(round(raw_box[0])), w - 1))\n        by1 = max(0, min(int(round(raw_box[1])), h - 1))\n        bx2 = max(bx1 + 1, min(int(round(raw_box[2])), w))\n        by2 = max(by1 + 1, min(int(round(raw_box[3])), h))\n\n        cv2.rectangle(frame, (bx1, by1), (bx2, by2), BOX_COLOR, BOX_THICKNESS)\n\n        label = f"ID {det.track_id}" if det.track_id is not None else f"{det.confidence:.2f}"\n        (text_w, text_h), baseline = cv2.getTextSize(label, FONT, config.FONT_SCALE, 1)\n\n        label_y1 = max(by1 - text_h - baseline - 4, 0)\n        cv2.rectangle(\n            frame,\n            (bx1, label_y1),\n            (min(bx1 + text_w + 4, w), min(label_y1 + text_h + baseline + 4, h)),\n            BOX_COLOR,\n            thickness=-1,\n        )\n        cv2.putText(\n            frame,\n            label,\n            (bx1 + 2, min(label_y1 + text_h + 2, h - 2)),\n            FONT,\n            config.FONT_SCALE,\n            TEXT_COLOR,\n            1,\n            cv2.LINE_AA,\n        )\n    return frame\n\n\n\ndef draw_overlay(frame: np.ndarray, head_count: int, crowd_level: str) -> np.ndarray:\n    """Draw a semi-transparent info panel with head count and crowd level."""\n    h, w = frame.shape[:2]\n    panel_w, panel_h = 300, 80\n    overlay = frame.copy()\n    cv2.rectangle(overlay, (0, 0), (panel_w, panel_h), (0, 0, 0), thickness=-1)\n    frame = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)\n\n    level_color = get_crowd_level_color(crowd_level)\n\n    cv2.putText(\n        frame, f"Head Count: {head_count}", (10, 30),\n        FONT, config.OVERLAY_FONT_SCALE, (255, 255, 255), 2, cv2.LINE_AA,\n    )\n    cv2.putText(\n        frame, f"Crowd Level: {crowd_level}", (10, 62),\n        FONT, config.OVERLAY_FONT_SCALE, level_color, 2, cv2.LINE_AA,\n    )\n    return frame\n\n\ndef generate_density_heatmap(\n    frame_shape: Tuple[int, int],\n    all_detections: Sequence[Sequence["Detection"]],\n    blur_ksize: int = 51,\n) -> np.ndarray:\n    """\n    Build a BGR heatmap image (same size as the source frame) showing where\n    heads were most frequently detected across many frames (or a single\n    frame for images). Each detected head center contributes a small\n    "heat" blob; blobs are accumulated, normalized, then colorized.\n\n    Args:\n        frame_shape: (height, width) of the source image/video frame.\n        all_detections: a list of per-frame Detection lists. For a still\n            image this is just `[detections]`.\n        blur_ksize: Gaussian blur kernel size (odd number) used to spread\n            each point into a smooth blob.\n\n    Returns:\n        A BGR heatmap image ready to be blended with the source frame or\n        saved standalone.\n    """\n    h, w = frame_shape\n    accumulator = np.zeros((h, w), dtype=np.float32)\n\n    for detections in all_detections:\n        for det in detections:\n            cx = int((det.x1 + det.x2) / 2)\n            cy = int((det.y1 + det.y2) / 2)\n            if 0 <= cy < h and 0 <= cx < w:\n                accumulator[cy, cx] += 1.0\n\n    if accumulator.max() > 0:\n        ksize = blur_ksize if blur_ksize % 2 == 1 else blur_ksize + 1\n        accumulator = cv2.GaussianBlur(accumulator, (ksize, ksize), 0)\n        accumulator = accumulator / accumulator.max()\n\n    heat_uint8 = np.uint8(255 * accumulator)\n    heatmap = cv2.applyColorMap(heat_uint8, cv2.COLORMAP_JET)\n    return heatmap\n\n\ndef blend_heatmap_with_frame(frame: np.ndarray, heatmap: np.ndarray, alpha: float = 0.55) -> np.ndarray:\n    """Overlay a colorized heatmap on top of the original frame."""\n    return cv2.addWeighted(heatmap, alpha, frame, 1 - alpha, 0)\n\n\ndef export_frame_counts_to_csv(\n    csv_path: str,\n    per_frame_counts: Sequence[int],\n    fps: float = 25.0,\n    classification_mode: str = "auto",\n    max_capacity: Optional[int] = None,\n    auto_crowd_levels: Optional[Sequence[str]] = None,\n) -> str:\n    """\n    Write a CSV with columns: Frame Number, Head Count, Crowd Level,\n    Risk Level, Timestamp (seconds, derived from fps). Returns the path\n    written to.\n\n    BUG FIX: this used to always call get_crowd_level(count) / get_risk_level(count)\n    with NO capacity argument, which silently applied the fixed absolute\n    thresholds (LOW_CROWD_MAX / MODERATE_CROWD_MAX) regardless of the mode\n    selected in Settings. A confined space (e.g. a bus, a small room)\n    analyzed in "Automatic (density-based)" mode -- where the on-screen\n    dashboard correctly reported density-based levels -- could export a\n    CSV whose "Crowd Level" column disagreed with the dashboard, and a\n    "Manual" run with a small custom max_capacity would still get\n    exported using the generic default thresholds instead of that capacity.\n\n    Fix: the caller now tells this function which mode was actually used.\n      - classification_mode == "auto": prefer the exact, already-computed\n        per-frame density-based levels via `auto_crowd_levels` (the same\n        values shown in the live dashboard / Analytics donut chart), so\n        the CSV always matches what the user saw on screen. Falls back to\n        recomputing from count only if that list isn\'t available.\n      - classification_mode == "manual": use `max_capacity` so a small,\n        confined space is classified relative to ITS capacity instead of\n        the generic global thresholds.\n    """\n    import csv as csv_module\n\n    ensure_dir(os.path.dirname(csv_path) or ".")\n\n    use_manual = classification_mode == "manual"\n    has_precomputed_auto = auto_crowd_levels is not None and len(auto_crowd_levels) == len(per_frame_counts)\n\n    with open(csv_path, "w", newline="", encoding="utf-8") as f:\n        writer = csv_module.writer(f)\n        writer.writerow(["Frame Number", "Head Count", "Crowd Level", "Risk Level", "Timestamp (s)"])\n        for idx, count in enumerate(per_frame_counts, start=1):\n            timestamp = round(idx / fps, 2) if fps > 0 else 0.0\n\n            if use_manual:\n                crowd_level = get_crowd_level(count, max_capacity=max_capacity)\n                risk_level = get_risk_level(count, max_capacity=max_capacity)\n            elif has_precomputed_auto:\n                crowd_level = auto_crowd_levels[idx - 1]\n                risk_level = {"Low Crowd": "Safe", "Moderate Crowd": "Warning", "High Crowd": "Critical"}[crowd_level]\n            else:\n                crowd_level = get_crowd_level(count)\n                risk_level = get_risk_level(count)\n\n            writer.writerow([idx, count, crowd_level, risk_level, timestamp])\n    return csv_path\n\n\n# --------------------------------------------------------------------------- #\n# Misc helpers\n# --------------------------------------------------------------------------- #\n\ndef ensure_dir(path: str) -> str:\n    """Create directory if it doesn\'t already exist. Returns the path."""\n    os.makedirs(path, exist_ok=True)\n    return path\n\n\ndef format_time(seconds: float) -> str:\n    """Format seconds into a human readable mm:ss.ms string."""\n    minutes = int(seconds // 60)\n    secs = seconds - minutes * 60\n    return f"{minutes:02d}:{secs:05.2f}"\n\n\nclass Timer:\n    """Simple context-manager based timer for measuring processing time."""\n\n    def __enter__(self) -> "Timer":\n        self._start = time.perf_counter()\n        self.elapsed = 0.0\n        return self\n\n    def __exit__(self, exc_type, exc_val, exc_tb) -> None:\n        self.elapsed = time.perf_counter() - self._start\n\n\ndef get_video_writer(\n    output_path: str,\n    fps: float,\n    frame_size: Tuple[int, int],\n    fourcc_str: str = "mp4v",\n) -> cv2.VideoWriter:\n    """Create an OpenCV VideoWriter with sane defaults for mp4 output."""\n    fourcc = cv2.VideoWriter_fourcc(*fourcc_str)\n    writer = cv2.VideoWriter(output_path, fourcc, fps, frame_size)\n    if not writer.isOpened():\n        raise IOError(f"Failed to open VideoWriter for path: {output_path}")\n    return writer\n\n\ndef resolve_class_ids(\n    model_names: dict, target_names: Sequence[str] = ("head",), allow_fallback: bool = True,\n) -> Optional[list]:\n    """\n    Resolve class-name(s) to class-id(s) using the model\'s `names` dict.\n\n    If none of `target_names` match the model\'s actual class names, this\n    used to fall back to `None` ("use all classes" for Ultralytics\'\n    `classes=` filter). That fallback is dangerous for THIS app: this\n    project only ever wants head/person detection, never general object\n    detection, so silently detecting every class the model knows (chairs,\n    laptops, bottles, etc. for a COCO-style checkpoint) produces exactly\n    the "boxes on furniture" bug -- wrong, misaligned-looking boxes that\n    have nothing to do with heads.\n\n    Instead: if there\'s no name match AND `allow_fallback` is True, fall\n    back to class id 0. The overwhelming majority of custom head/person/\n    crowd-detection exports (Ultralytics `yolo export`, Roboflow, CVAT,\n    etc.) are single-class with the target class as id 0 regardless of\n    what it\'s literally named in `model.names` -- so this is a much safer\n    default than "everything."\n\n    `allow_fallback=False` is for callers that need a STRICT match (e.g.\n    "does this model have a real, explicitly named \'head\' class?" --\n    used to decide native-head vs person-with-estimation drawing mode.\n    Falling back to id 0 there would wrongly mark every model as a native\n    head model, since id 0 always exists.\n    """\n    name_to_id = {v.lower(): k for k, v in model_names.items()}\n    ids = [name_to_id[n.lower()] for n in target_names if n.lower() in name_to_id]\n    if ids:\n        return ids\n    if allow_fallback:\n        raise ValueError("This checkpoint has no head/person class. Select the documented crowd model.")\n    return None\n\n\ndef is_stock_pretrained_checkpoint(model_names: dict) -> bool:\n    """\n    Heuristic: True if `model_names` looks like an UNTOUCHED, plain\n    Ultralytics pretrained checkpoint (all 80 COCO classes) rather than a\n    model fine-tuned on a custom head/person dataset.\n\n    This matters because a stock checkpoint still "works" in this app --\n    it has a real \'person\' class, so detection + the head-crop heuristic\n    still run and can look fine at a glance -- but it was never trained on\n    YOUR head dataset, so its accuracy on your specific cameras/scenes is\n    whatever generic COCO training happened to generalize to, not\n    anything tuned for your use case. Silently treating it as "the head\n    model" is exactly what caused the earlier best.pt/model.pt mix-up.\n    """\n    names_lower = {v.lower() for v in model_names.values()}\n    coco_markers = {"chair", "laptop", "cell phone", "keyboard", "bicycle", "giraffe"}\n    return len(model_names) >= 20 and "person" in names_lower and len(names_lower & coco_markers) >= 3\n\n\ndef describe_model_classes(model_names: dict, target_names: Sequence[str] = ("person", "head")) -> str:\n    """\n    Human-readable one-liner describing whether a loaded model\'s classes\n    match what this app expects, for display in Settings so a mismatched\n    model (e.g. a generic/pretrained checkpoint dropped in by mistake) is\n    obvious instead of silently producing wrong-looking boxes.\n    """\n    if is_stock_pretrained_checkpoint(model_names):\n        return (\n            f"⚠️ This is the stock pretrained checkpoint ({len(model_names)} generic COCO "\n            f"classes: person, car, chair, laptop, ...) -- NOT fine-tuned on your head "\n            f"dataset. It will still detect \'person\' and draw estimated head boxes, but "\n            f"accuracy on your cameras is whatever generic COCO training happened to "\n            f"generalize to. Use train.py to fine-tune it on your dataset before treating "\n            f"it as a real head-detection model."\n        )\n    names_lower = {v.lower() for v in model_names.values()}\n    matched = [n for n in target_names if n.lower() in names_lower]\n    if matched:\n        return f"✅ Model classes match expected: {\', \'.join(matched)}."\n    class_list = ", ".join(f"{k}: {v}" for k, v in sorted(model_names.items()))\n    if len(model_names) == 1:\n        return (\n            f"⚠️ Model\'s single class isn\'t named \'person\'/\'head\' ({class_list}) -- "\n            f"assuming it\'s still the intended head/person class (id 0)."\n        )\n    return (\n        f"❌ None of this model\'s classes are \'person\' or \'head\' -- it has {len(model_names)} "\n        f"classes ({class_list}). This looks like a general-purpose/pretrained model, not a "\n        f"head-detection model, and will likely draw boxes on unrelated objects (chairs, "\n        f"laptops, etc). Double-check you selected the right weights file."\n    )\n\n\ndef is_valid_weights_reference(weights_path: str) -> bool:\n    """\n    True if `weights_path` is either:\n      - an existing file on disk (a real custom-trained .pt path), or\n      - a bare official model name with no directory component (e.g.\n        "yolov8n.pt") that Ultralytics resolves/downloads automatically.\n\n    A plain `os.path.isfile()` check would incorrectly reject official\n    pretrained model names on a machine where they haven\'t been\n    downloaded yet, since the file doesn\'t exist until YOLO() fetches it.\n    """\n    if os.path.isfile(weights_path):\n        return True\n    has_no_dir = os.path.dirname(weights_path) == ""\n    return has_no_dir and weights_path.lower().endswith(".pt")\n\n\ndef discover_weight_files() -> list:\n    """\n    Scan the project for candidate .pt model files so the UI can offer a\n    dropdown instead of requiring the user to type a path by hand.\n\n    Looks in:\n      - the project root (e.g. best.pt sitting next to app.py)\n      - runs/train/<run_name>/weights/*.pt for every training run\n        Ultralytics has produced (best.pt, last.pt, or any other .pt saved\n        there -- e.g. a manually renamed "model.pt")\n\n    Returns a list of dicts: [{"label": <friendly name>, "path": <abs path>}],\n    de-duplicated by absolute path, sorted with the configured default\n    weights first, then alphabetically by label.\n    """\n    candidates = {}\n\n    def _add(path: str, label: str) -> None:\n        abspath = os.path.abspath(path)\n        if os.path.isfile(abspath):\n            candidates[abspath] = label\n\n    # Project-root-level weights (e.g. best.pt shipped alongside app.py)\n    try:\n        for fname in os.listdir(config.PROJECT_ROOT):\n            if fname.lower().endswith(".pt"):\n                label = fname\n                if os.path.abspath(os.path.join(config.PROJECT_ROOT, fname)) == os.path.abspath(\n                    config.PRETRAINED_BASE_CHECKPOINT\n                ):\n                    label = f"{fname}  (stock pretrained -- not fine-tuned on your dataset)"\n                _add(os.path.join(config.PROJECT_ROOT, fname), label)\n    except OSError:\n        pass\n\n    # Every runs/train/<run_name>/weights/*.pt\n    train_root = config.TRAIN_PROJECT_DIR\n    if os.path.isdir(train_root):\n        for run_name in sorted(os.listdir(train_root)):\n            weights_dir = os.path.join(train_root, run_name, "weights")\n            if not os.path.isdir(weights_dir):\n                continue\n            try:\n                for fname in sorted(os.listdir(weights_dir)):\n                    if fname.lower().endswith(".pt"):\n                        stem = os.path.splitext(fname)[0]\n                        _add(os.path.join(weights_dir, fname), f"{run_name} / {stem}")\n            except OSError:\n                pass\n\n    default_path = os.path.abspath(config.BEST_WEIGHTS_PATH)\n    items = [{"label": label, "path": path} for path, label in candidates.items()]\n    items.sort(key=lambda d: (d["path"] != default_path, d["label"].lower()))\n    return items\n\n\ndef estimate_head_box(\n    x1: int, y1: int, x2: int, y2: int,\n    top_fraction: float = 0.22,\n    width_shrink: float = 0.15,\n) -> Tuple[int, int, int, int]:\n    """\n    Estimate a head bounding box from a full-body person bounding box.\n\n    Used when the loaded model only knows "person" (e.g. pretrained COCO\n    yolov8n.pt) and has no dedicated "head" class -- the top portion of\n    the person box, narrowed horizontally, approximates the head region.\n\n    Args:\n        x1, y1, x2, y2: the person\'s full bounding box.\n        top_fraction: fraction of the person box height counted as head.\n        width_shrink: fraction shaved off each side (shoulders are wider\n            than the head).\n\n    Returns:\n        (hx1, hy1, hx2, hy2) estimated head box, guaranteed valid\n        (hx2 > hx1, hy2 > hy1).\n    """\n    width = x2 - x1\n    height = y2 - y1\n    if width <= 0 or height <= 0:\n        return x1, y1, x2, y2\n\n    head_height = max(1, int(round(height * top_fraction)))\n    dx = int(round(width * width_shrink))\n\n    hx1 = x1 + dx\n    hx2 = x2 - dx\n    hy1 = y1\n    hy2 = y1 + head_height\n\n    # Guard against over-shrinking on very narrow boxes.\n    if hx2 <= hx1:\n        hx1, hx2 = x1, x2\n    if hy2 <= hy1:\n        hy2 = hy1 + 1\n\n    return hx1, hy1, hx2, hy2\n\n\ndef estimate_body_box_from_head(\n    x1: int, y1: int, x2: int, y2: int,\n    frame_width: int,\n    frame_height: int,\n    height_multiplier: float = 2.3,\n    width_multiplier: float = 1.7,\n) -> Tuple[int, int, int, int]:\n    """\n    Estimate a body-sized bounding box from a HEAD bounding box.\n\n    Needed when the loaded model natively detects HEADS ONLY (e.g. a\n    head-trained YOLO11 model, `model.names == {0: "head"}`), which is\n    great for accurate small/distant-person counting but produces tiny\n    boxes that look wrong for a crowd-management display expecting\n    larger, person-sized boxes.\n\n    NOTE: this is intentionally NOT the mathematical inverse of\n    estimate_head_box() above. That function\'s ratios describe a\n    STANDING person\'s full-body proportions (head ~= 22% of total height),\n    which is the wrong model for typical office/desk CCTV footage where\n    most people are SEATED and only visible from the head down to roughly\n    the chest/shoulders before a desk or chair occludes the rest --\n    inverting the standing-body ratio there produces boxes that extend\n    well past the person into whatever is behind them. `height_multiplier`\n    / `width_multiplier` here instead approximate "head + neck + shoulders\n    + upper chest", a conservative box that stays visually anchored to the\n    person regardless of whether they\'re seated or standing (see\n    config.HEAD_TO_BODY_HEIGHT_MULTIPLIER / _WIDTH_MULTIPLIER to tune).\n\n    This is a DISPLAY-ONLY estimate: it is intentionally not used for\n    head-count or packing-density calculations, which stay based on the\n    real, accurate head box (inflating those to body-sized boxes would\n    make the auto density/crowd-level classifier -- calibrated on head-box\n    sizes -- wildly overestimate crowding).\n\n    Args:\n        x1, y1, x2, y2: the detected head\'s bounding box.\n        frame_width, frame_height: dimensions of the source frame, so the\n            estimated body box can be clipped to stay on-screen.\n        height_multiplier: body box height = head box height * this.\n        width_multiplier: body box width = head box width * this.\n\n    Returns:\n        (bx1, by1, bx2, by2) estimated body box, clipped to the frame and\n        guaranteed valid (bx2 > bx1, by2 > by1).\n    """\n    head_w = max(x2 - x1, 1)\n    head_h = max(y2 - y1, 1)\n\n    body_height = head_h * height_multiplier\n    body_width = head_w * width_multiplier\n\n    # Body is centered on the same horizontal midpoint as the head, and\n    # extends downward starting at the top of the head.\n    cx = (x1 + x2) / 2.0\n    bx1 = int(round(cx - body_width / 2.0))\n    bx2 = int(round(cx + body_width / 2.0))\n    by1 = y1\n    by2 = int(round(y1 + body_height))\n\n    # Clip to the frame so an estimated body near an edge doesn\'t draw\n    # off-screen.\n    bx1 = max(0, min(bx1, frame_width - 1))\n    bx2 = max(bx1 + 1, min(bx2, frame_width))\n    by1 = max(0, min(by1, frame_height - 1))\n    by2 = max(by1 + 1, min(by2, frame_height))\n\n    return bx1, by1, bx2, by2\n\n\ndef is_plausible_head_box(\n    x1: int, y1: int, x2: int, y2: int,\n    min_aspect: float = 0.35,\n    max_aspect: float = 2.2,\n) -> bool:\n    """\n    Sanity-check a raw detection\'s aspect ratio (width / height) against\n    what a real human head can plausibly look like, regardless of viewing\n    angle (roughly circular/oval -> close to 1:1, with some allowance for\n    tilt, hair, or a side profile).\n\n    Needed because a native head-only model can occasionally fire on a\n    non-head object that happens to clear the confidence threshold (a\n    hand resting on a laptop, a chair silhouette, etc.) -- these tend to\n    produce boxes with an aspect ratio far outside anything a head can\n    have (e.g. ~4:1, wide and short), even at high confidence. Confidence\n    alone does not catch this, since the model can be very confident about\n    a wrong detection; shape does.\n\n    Deliberately permissive by default (0.35-2.2) so it only rejects clear\n    outliers, not merely unusual real heads (tilted, partially cropped,\n    side-on, etc.).\n    """\n    w = max(x2 - x1, 1)\n    h = max(y2 - y1, 1)\n    aspect = w / h\n    return min_aspect <= aspect <= max_aspect\n\n\n# --------------------------------------------------------------------------- #\n# Dataset validation / cache handling (training bug fix)\n# --------------------------------------------------------------------------- #\n\n_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")\n\n\ndef _images_dir_to_labels_dir(images_dir: str) -> str:\n    """\n    Mirror Ultralytics\' own path convention: an images directory is mapped\n    to its labels directory by replacing the LAST path segment named\n    \'images\' with \'labels\'. Works regardless of OS path separator.\n    """\n    norm = images_dir.replace("\\\\", "/").rstrip("/")\n    parts = norm.split("/")\n    for i in range(len(parts) - 1, -1, -1):\n        if parts[i] == "images":\n            parts[i] = "labels"\n            return "/".join(parts)\n    # Fallback: no \'images\' segment found, assume labels dir is a sibling.\n    return norm\n\n\ndef _load_data_yaml(data_yaml_path: str) -> dict:\n    import yaml  # local import: keeps this optional dependency out of the hot path\n\n    if not os.path.isfile(data_yaml_path):\n        raise FileNotFoundError(f"data.yaml not found at: {data_yaml_path}")\n    with open(data_yaml_path, "r", encoding="utf-8") as f:\n        return yaml.safe_load(f)\n\n\ndef clear_dataset_cache(data_yaml_path: str, logger: Optional[logging.Logger] = None) -> None:\n    """\n    Delete stale Ultralytics .cache files (train.cache / val.cache) for the\n    splits referenced in data.yaml. Ultralytics trusts an existing cache\n    file blindly and will NOT re-scan labels on disk if a (possibly stale\n    or wrong) cache is present -- this forces a fresh scan.\n    """\n    log = logger or setup_logger(__name__)\n    data_cfg = _load_data_yaml(data_yaml_path)\n    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))\n\n    for split in ("train", "val"):\n        images_dir = data_cfg.get(split)\n        if not images_dir:\n            continue\n        if not os.path.isabs(images_dir):\n            images_dir = os.path.join(yaml_dir, images_dir)\n\n        labels_dir = _images_dir_to_labels_dir(images_dir)\n        cache_path = labels_dir.rstrip("/").replace("/", os.sep) + ".cache"\n\n        if os.path.isfile(cache_path):\n            try:\n                os.remove(cache_path)\n                log.info("Removed stale cache: %s", cache_path)\n            except OSError as exc:\n                log.warning("Could not remove cache file %s: %s", cache_path, exc)\n\n\ndef _parse_label_file(label_path: str, nc: int) -> dict:\n    """\n    Parse a single YOLO-format label .txt file the way Ultralytics does,\n    and report exactly why rows are or aren\'t counted as valid.\n\n    Returns a dict: {\n        \'valid_boxes\': int,          # rows that pass every check\n        \'total_rows\': int,           # non-blank lines in the file\n        \'malformed_rows\': int,       # wrong token count / non-numeric\n        \'class_out_of_range\': int,   # class id not in [0, nc-1]\n        \'bad_coords\': int,           # coords outside [0,1] or w/h <= 0\n        \'reasons\': list[str],        # human-readable first few problems\n    }\n    """\n    result = {\n        "valid_boxes": 0,\n        "total_rows": 0,\n        "malformed_rows": 0,\n        "class_out_of_range": 0,\n        "bad_coords": 0,\n        "duplicate_rows": 0,\n        "reasons": [],\n    }\n\n    try:\n        with open(label_path, "r", encoding="utf-8") as f:\n            lines = [ln.strip() for ln in f.readlines() if ln.strip()]\n    except (OSError, UnicodeDecodeError) as exc:\n        result["reasons"].append(f"could not read file: {exc}")\n        return result\n\n    seen_boxes = set()\n    for line_no, line in enumerate(lines, start=1):\n        result["total_rows"] += 1\n        tokens = line.split()\n\n        if len(tokens) != 5:\n            result["malformed_rows"] += 1\n            if len(result["reasons"]) < 3:\n                result["reasons"].append(\n                    f"line {line_no}: expected 5 values (class x y w h), got {len(tokens)}: \'{line}\'"\n                )\n            continue\n\n        try:\n            class_value = float(tokens[0])\n            if not np.isfinite(class_value) or not class_value.is_integer():\n                raise ValueError(\'Class must be a finite integer\')\n            class_id = int(class_value)\n            x, y, w, h = (float(t) for t in tokens[1:])\n        except ValueError:\n            result["malformed_rows"] += 1\n            if len(result["reasons"]) < 3:\n                result["reasons"].append(f"line {line_no}: non-numeric value(s): \'{line}\'")\n            continue\n\n        if class_id < 0 or class_id >= nc:\n            result["class_out_of_range"] += 1\n            if len(result["reasons"]) < 3:\n                result["reasons"].append(\n                    f"line {line_no}: class id {class_id} is outside valid range [0, {nc - 1}] "\n                    f"(data.yaml declares nc={nc})"\n                )\n            continue\n\n        # Ultralytics requires normalized coords in (0, 1] and positive width/height.\n        if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):\n            result["bad_coords"] += 1\n            if len(result["reasons"]) < 3:\n                result["reasons"].append(f"line {line_no}: x/y center out of [0,1] range: \'{line}\'")\n            continue\n        if not np.isfinite([x,y,w,h]).all() or w <= 0.0 or h <= 0.0 or w > 1.0 or h > 1.0:\n            result["bad_coords"] += 1\n            if len(result["reasons"]) < 3:\n                result["reasons"].append(f"line {line_no}: invalid width/height: \'{line}\'")\n            continue\n\n        box=(class_id,x,y,w,h)\n        if box in seen_boxes:\n            result[\'duplicate_rows\'] += 1\n            continue\n        seen_boxes.add(box)\n        result["valid_boxes"] += 1\n\n    return result\n\n\ndef log_resolved_dataset_paths(data_yaml_path: str, logger: Optional[logging.Logger] = None) -> None:\n    """\n    Print the exact, fully-resolved image/label directories that will be\n    used for this run, for debugging path-resolution issues before training\n    starts (see normalize_data_yaml() for why this matters on Windows).\n    """\n    log = logger or setup_logger(__name__)\n    data_cfg = _load_data_yaml(data_yaml_path)\n    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))\n\n    log.info("Resolved data.yaml: %s", os.path.abspath(data_yaml_path))\n    for split in ("train", "val"):\n        raw = data_cfg.get(split)\n        if not raw:\n            continue\n        images_dir = raw if os.path.isabs(raw) else os.path.join(yaml_dir, raw)\n        images_dir = os.path.normpath(images_dir)\n        labels_dir = os.path.normpath(_images_dir_to_labels_dir(images_dir).replace("/", os.sep))\n        log.info("[%s] raw path in yaml : %s", split, raw)\n        log.info("[%s] resolved images  : %s", split, images_dir)\n        log.info("[%s] resolved labels  : %s", split, labels_dir)\n\n\ndef normalize_data_yaml(data_yaml_path: str, logger: Optional[logging.Logger] = None) -> str:\n    """\n    Root-cause fix for the misleading "All labels empty in .../train.cache"\n    error on Windows.\n\n    Ultralytics resolves each image\'s label path with a hardcoded,\n    OS-specific string substitution:\n        f"{os.sep}images{os.sep}"  ->  f"{os.sep}labels{os.sep}"\n    On Windows, os.sep == \'\\\\\'. If data.yaml specifies paths with forward\n    slashes (e.g. "E:/project/dataset/images/train"), that substring never\n    matches a \'\\\\images\\\\\' pattern, so Ultralytics silently fails to locate\n    ANY label file for ANY image -- even when the images and labels are\n    100% correctly paired on disk. It then reports every image as having\n    an empty label set.\n\n    This function reads the user\'s original data.yaml (never modified),\n    resolves the train/val paths to absolute paths using the current OS\'s\n    separator, and writes a normalized copy to a temp file for Ultralytics\n    to consume. If no normalization was actually needed, the original path\n    is returned unchanged.\n    """\n    import tempfile\n    import yaml\n\n    log = logger or setup_logger(__name__)\n    data_cfg = _load_data_yaml(data_yaml_path)\n    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))\n\n    changed = False\n    for split in ("train", "val", "test"):\n        raw = data_cfg.get(split)\n        if not raw:\n            continue\n        abs_path = raw if os.path.isabs(raw) else os.path.join(yaml_dir, raw)\n        normalized = os.path.normpath(abs_path)\n        if normalized != raw:\n            changed = True\n        data_cfg[split] = normalized\n\n    if not changed:\n        log.info("data.yaml paths already OS-normalized -- using original file.")\n        return data_yaml_path\n\n    tmp_path = os.path.join(tempfile.gettempdir(), "crowd_mgmt_data_normalized.yaml")\n    with open(tmp_path, "w", encoding="utf-8") as f:\n        yaml.safe_dump(data_cfg, f, sort_keys=False)\n\n    log.info(\n        "data.yaml paths were not OS-normalized (likely mixed \'/\' separators on "\n        "Windows) -- Ultralytics\' internal images->labels path substitution "\n        "requires exact os.sep matches, or it silently can\'t find your labels. "\n        "Wrote a normalized copy for training (your original data.yaml is untouched): %s",\n        tmp_path,\n    )\n    return tmp_path\n\n\ndef validate_dataset(data_yaml_path: str, logger: Optional[logging.Logger] = None) -> None:\n    """\n    Verify that images and labels are correctly paired AND that label files\n    actually contain valid, parseable bounding boxes, for every split\n    listed in data.yaml. This mirrors the checks Ultralytics performs\n    internally, so a broken dataset fails HERE with a specific, actionable\n    reason instead of surfacing later as the opaque\n    "All labels empty in ...cache" ValueError from Ultralytics.\n    """\n    log = logger or setup_logger(__name__)\n    data_cfg = _load_data_yaml(data_yaml_path)\n    yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))\n\n    nc = data_cfg.get("nc")\n    names = data_cfg.get("names")\n    if nc is None and isinstance(names, dict):\n        nc = len(names)\n    if nc is None:\n        raise ValueError(f"data.yaml at \'{data_yaml_path}\' has no \'nc\' (or \'names\') field -- cannot validate.")\n\n    log.info("Resolved data.yaml: %s", os.path.abspath(data_yaml_path))\n    log.info("nc=%s names=%s", nc, names)\n\n    for split in ("train", "val"):\n        images_dir = data_cfg.get(split)\n        if not images_dir:\n            log.warning("data.yaml has no \'%s\' entry -- skipping validation for it.", split)\n            continue\n        if not os.path.isabs(images_dir):\n            images_dir = os.path.join(yaml_dir, images_dir)\n        images_dir = os.path.normpath(images_dir)\n\n        labels_dir = os.path.normpath(_images_dir_to_labels_dir(images_dir).replace("/", os.sep))\n\n        log.info("[%s] images_dir = %s", split, images_dir)\n        log.info("[%s] labels_dir = %s", split, labels_dir)\n\n        if not os.path.isdir(images_dir):\n            raise ValueError(f"[{split}] Images directory does not exist: {images_dir}")\n        if not os.path.isdir(labels_dir):\n            raise ValueError(f"[{split}] Labels directory does not exist: {labels_dir}")\n\n        image_files = [\n            f for f in os.listdir(images_dir)\n            if f.lower().endswith(_IMAGE_EXTENSIONS)\n        ]\n        label_files_on_disk = {\n            f for f in os.listdir(labels_dir) if f.lower().endswith(".txt")\n        }\n        label_stems = {os.path.splitext(f)[0] for f in label_files_on_disk}\n\n        if not image_files:\n            raise ValueError(f"[{split}] No image files found in: {images_dir}")\n\n        matched, unmatched = 0, []\n        total_valid_boxes = 0\n        files_with_zero_valid_boxes = []\n        malformed_total = 0\n        class_range_total = 0\n        bad_coord_total = 0\n        first_bad_example = None\n\n        for img_name in image_files:\n            stem = os.path.splitext(img_name)[0]\n            if stem not in label_stems:\n                unmatched.append(img_name)\n                continue\n            matched += 1\n\n            label_path = os.path.join(labels_dir, stem + ".txt")\n            stats = _parse_label_file(label_path, nc=nc)\n            if stats[\'duplicate_rows\']:\n                raise ValueError(f\'[{split}] {stem}.txt contains {stats["duplicate_rows"]} duplicate boxes. Repair annotations before training.\')\n            if stats[\'malformed_rows\'] or stats[\'bad_coords\'] or stats[\'class_out_of_range\']:\n                raise ValueError(f\'[{split}] Invalid labels in {stem}.txt: {stats["reasons"]}\')\n\n            total_valid_boxes += stats["valid_boxes"]\n            malformed_total += stats["malformed_rows"]\n            class_range_total += stats["class_out_of_range"]\n            bad_coord_total += stats["bad_coords"]\n\n            if stats["valid_boxes"] == 0:\n                files_with_zero_valid_boxes.append(stem)\n                if first_bad_example is None and stats["reasons"]:\n                    first_bad_example = (stem, stats)\n\n        log.info(\n            "[%s] images=%d matched_label_files=%d unmatched=%d",\n            split, len(image_files), matched, len(unmatched),\n        )\n        log.info(\n            "[%s] total_valid_boxes=%d files_with_zero_valid_boxes=%d "\n            "malformed_rows=%d class_id_out_of_range=%d bad_coords=%d",\n            split, total_valid_boxes, len(files_with_zero_valid_boxes),\n            malformed_total, class_range_total, bad_coord_total,\n        )\n\n        if unmatched[:5]:\n            log.warning("[%s] Example images with NO matching label file: %s", split, unmatched[:5])\n\n        if matched == 0:\n            raise ValueError(\n                f"[{split}] 0 of {len(image_files)} images have a matching label file in "\n                f"\'{labels_dir}\'. Every image \'X.jpg\' needs a same-named \'X.txt\' in the "\n                f"labels folder. Fix the pairing before training."\n            )\n\n        if total_valid_boxes == 0:\n            detail = ""\n            if first_bad_example:\n                stem, stats = first_bad_example\n                detail = (\n                    f"\\nExample failing file: {stem}.txt\\n"\n                    f"  total_rows={stats[\'total_rows\']} malformed={stats[\'malformed_rows\']} "\n                    f"class_out_of_range={stats[\'class_out_of_range\']} bad_coords={stats[\'bad_coords\']}\\n"\n                    f"  reasons: {stats[\'reasons\']}"\n                )\n            raise ValueError(\n                f"[{split}] Labels and images are correctly paired ({matched} files matched), "\n                f"but ZERO valid bounding boxes were found across all {matched} label files. "\n                f"This is the actual cause of Ultralytics\' \'All labels empty\' error -- it is a "\n                f"label CONTENT problem, not a pairing/cache problem.\\n"\n                f"Breakdown across {matched} files: malformed_rows={malformed_total}, "\n                f"class_id_out_of_range={class_range_total} (valid range is 0-{nc - 1}), "\n                f"invalid_coords_or_size={bad_coord_total}."\n                f"{detail}"\n            )\n\n        if files_with_zero_valid_boxes:\n            log.warning(\n                "[%s] %d/%d matched label files have ZERO valid boxes (background images). "\n                "This is normal in small numbers, but check if this count looks unexpectedly high.",\n                split, len(files_with_zero_valid_boxes), matched,\n            )\n', 'src/utils/privacy.py': '"""Credential redaction shared by persisted records and reports."""\nimport re\n\ndef redact_sensitive(value):\n    if isinstance(value, dict):\n        return {k: (\'[redacted]\' if re.search(r\'password|token|secret\', str(k), re.I) else redact_sensitive(v)) for k,v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [redact_sensitive(v) for v in value]\n    if isinstance(value, str):\n        value = re.sub(r\'(?i)(rtsp[s]?|https?)://[^\\s/@]+(?::[^\\s/@]*)?@\', r\'\\1://[redacted]@\', value)\n        return re.sub(r\'(?i)([?&](?:password|passwd|token|api_key|secret)=)[^&\\s]*\', r\'\\1[redacted]\', value)\n    return value\n', 'src/utils/__init__.py': '"""\nsrc.utils package initialization\n"""\nfrom src.utils.helpers import *  # noqa: F401, F403\n', 'config.py': '"""Central paths and defaults. The app reports configured occupancy, not safety predictions."""\nfrom __future__ import annotations\nimport os\nPROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))\nPROJECT_DIR = os.path.dirname(PROJECT_ROOT)\nDATA_YAML_PATH = os.environ.get(\'CROWD_DATA_YAML\', os.path.join(PROJECT_ROOT, \'dataset\', \'data.yaml\'))\nTRAIN_PROJECT_DIR = os.path.join(PROJECT_ROOT,\'runs\',\'train\')\nTRAIN_RUN_NAME = \'crowd_head_experiment\'\nCUSTOM_WEIGHTS_PATH = os.path.join(TRAIN_PROJECT_DIR,TRAIN_RUN_NAME,\'weights\',\'best.pt\')\nBEST_WEIGHTS_PATH = os.environ.get(\'CROWD_MODEL_PATH\',os.path.join(PROJECT_ROOT,\'models\',\'head_detector.pt\'))\nPRETRAINED_BASE_CHECKPOINT = os.path.join(PROJECT_ROOT,\'yolo11n.pt\')\nTRAIN_BASE_MODEL = \'yolo11n.pt\'\nTRAIN_EPOCHS = 50\nTRAIN_IMGSZ = 512\nTRAIN_BATCH = 4\nTRAIN_WORKERS = 2\nTRAIN_PATIENCE = 15\nTRAIN_DEVICE = \'cpu\'\nINFERENCE_CONF_THRESHOLD = .35\nINFERENCE_IOU_THRESHOLD = .5\nINFERENCE_IMGSZ = 640\nINFERENCE_DEVICE = \'cpu\'\nTARGET_CLASS_NAMES = (\'person\',\'head\')\nNATIVE_HEAD_CLASS_NAMES = (\'head\',)\nHEAD_TOP_FRACTION = .22\nHEAD_WIDTH_SHRINK = .15\nHEAD_TO_BODY_HEIGHT_MULTIPLIER = 2.3\nHEAD_TO_BODY_WIDTH_MULTIPLIER = 1.7\nSHOW_BODY_BOX_ESTIMATE = False\nENABLE_TRACKING = False\nENABLE_BOX_SMOOTHING = False\nBOX_SMOOTHING_ALPHA = .45\nLOW_CROWD_MAX = 20\nMODERATE_CROWD_MAX = 40\nCROWD_LEVEL_COLORS_BGR = {\'Low Crowd\':(0,200,0),\'Moderate Crowd\':(0,165,255),\'High Crowd\':(0,0,255)}\nBOX_COLOR_BGR = (255,120,40)\nBOX_THICKNESS = 2\nTEXT_COLOR_BGR = (255,255,255)\nFONT_SCALE = .5\nOVERLAY_FONT_SCALE = .7\nASSUMED_MAX_CAPACITY = 80\n# Backward-compatible helper constants. The current UI does not display risk predictions.\nRISK_LOW_MAX = LOW_CROWD_MAX\nRISK_MODERATE_MAX = MODERATE_CROWD_MAX\nRISK_LEVEL_ICONS = {\'Safe\':\'Low\',\'Warning\':\'Moderate\',\'Critical\':\'High\'}\nALLOWED_IMAGE_TYPES = [\'jpg\',\'jpeg\',\'png\',\'bmp\']\nALLOWED_VIDEO_TYPES = [\'mp4\',\'avi\',\'mov\',\'mkv\']\nOUTPUTS_ROOT = os.path.join(PROJECT_ROOT,\'outputs\')\nOUTPUT_IMAGES_DIR = os.path.join(OUTPUTS_ROOT,\'images\')\nSTREAMLIT_OUTPUT_DIR = os.path.join(OUTPUTS_ROOT,\'streamlit_images\')\nSTREAMLIT_VIDEO_OUTPUT_DIR = os.path.join(OUTPUTS_ROOT,\'streamlit_videos\')\nHISTORY_DIR = os.path.join(OUTPUTS_ROOT,\'history\')\nHISTORY_FILE = os.path.join(HISTORY_DIR,\'history.json\')\nREPORTS_DIR = os.path.join(OUTPUTS_ROOT,\'reports\')\nCSV_EXPORT_DIR = os.path.join(OUTPUTS_ROOT,\'csv\')\nHEATMAP_DIR = os.path.join(OUTPUTS_ROOT,\'heatmaps\')\nMAX_HISTORY_ENTRIES = 200\nPROJECT_DISPLAY_NAME = \'CrowdVision Campus — Occupancy and Flow Review\'\nPROJECT_VERSION = \'4.0.0\'\n', 'scripts/prepare_scut.py': '"""Convert the official SCUT-HEAD Part B archive to YOLO, preserving its splits.\n\nAcademic research only. Obtain the archive from HCIILAB/SCUT-HEAD-Dataset-Release.\nRaw images, XML and generated labels are local data, not repository content.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport zipfile\nimport xml.etree.ElementTree as ET\nimport cv2\nimport numpy as np\n\ndef convert(archive, output):\n    output=Path(output)\n    if output.exists(): raise FileExistsError(\'Choose a new output folder.\')\n    output.mkdir(parents=True)\n    prefix=\'SCUT_HEAD_Part_B/\'\n    summary={}; seen=set(); corrections=0; missing_sizes=[]\n    with zipfile.ZipFile(archive) as z:\n        image_names={Path(n).stem:n for n in z.namelist() if \'/JPEGImages/\' in n and n.lower().endswith((\'.jpg\',\'.jpeg\',\'.png\'))}\n        for split in (\'train\',\'val\',\'test\'):\n            ids=z.read(prefix+f\'ImageSets/Main/{split}.txt\').decode().split()\n            images=output/\'images\'/split; labels=output/\'labels\'/split\n            images.mkdir(parents=True); labels.mkdir(parents=True)\n            count=0\n            for name in ids:\n                if name in seen: raise ValueError(\'Overlapping official splits: \'+name)\n                seen.add(name)\n                raw=z.read(image_names[name]); frame=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)\n                if frame is None: raise ValueError(\'Unreadable image \'+name)\n                h,w=frame.shape[:2]\n                xml=ET.fromstring(z.read(prefix+\'Annotations/\'+name+\'.xml\'))\n                declared=(int(xml.findtext(\'size/width\')),int(xml.findtext(\'size/height\')))\n                if declared == (0,0):\n                    missing_sizes.append(name)\n                elif declared != (w,h):\n                    raise ValueError(\'Image/XML dimension mismatch: \'+name)\n                boxes=[]\n                for obj in xml.findall(\'object\'):\n                    b=obj.find(\'bndbox\')\n                    x1,y1,x2,y2=[float(b.findtext(k)) for k in (\'xmin\',\'ymin\',\'xmax\',\'ymax\')]\n                    # VOC has one-based inclusive coordinates; clamp documented out-of-image boxes.\n                    original=(x1-1,y1-1,x2,y2)\n                    x1,y1,x2,y2=max(0,x1-1),max(0,y1-1),min(w,x2),min(h,y2)\n                    if (x1,y1,x2,y2)!=original: corrections+=1\n                    if x2<=x1 or y2<=y1: raise ValueError(\'Degenerate box: \'+name)\n                    boxes.append(f\'0 {(x1+x2)/(2*w):.8f} {(y1+y2)/(2*h):.8f} {(x2-x1)/w:.8f} {(y2-y1)/h:.8f}\')\n                if len(boxes)!=len(set(boxes)): raise ValueError(\'Duplicate original boxes: \'+name)\n                (images/(name+Path(image_names[name]).suffix.lower())).write_bytes(raw)\n                (labels/(name+\'.txt\')).write_text(\'\\n\'.join(boxes),encoding=\'utf-8\')\n                count+=len(boxes)\n            summary[split]=dict(images=len(ids),heads=count)\n    summary.update(source=\'https://github.com/HCIILAB/SCUT-HEAD-Dataset-Release\',\n                   license=\'Academic research only\',archive_sha256=hashlib.sha256(Path(archive).read_bytes()).hexdigest(),\n                   clamped_boxes=corrections,missing_xml_sizes_read_from_image=missing_sizes,\n                   conversion=\'VOC one-based inclusive to clipped YOLO head boxes\')\n    (output/\'provenance.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n    print(json.dumps(summary,indent=2))\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(); parser.add_argument(\'archive\'); parser.add_argument(\'output\')\n    args=parser.parse_args(); convert(args.archive,args.output)\n', 'scripts/model_experiment.py': '"""Reproducible CPU fine-tuning pilot and count-error promotion gate.\n\nLocal dataset remains private. A new checkpoint is never installed automatically.\nThe existing model\'s training provenance is unknown: this is a local comparison,\nnot proof of generalization to unseen college cameras.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport random\nimport sys\nimport time\nimport os\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nos.environ.setdefault(\'OMP_NUM_THREADS\', \'4\')\nos.environ.setdefault(\'MKL_NUM_THREADS\', \'4\')\nimport cv2\nimport numpy as np\nimport torch\nimport yaml\nfrom ultralytics import YOLO\nfrom src.core.detector import detect_heads_in_frame\nfrom src.services.storage import atomic_json\n\n\ndef audit(dataset):\n    splits, seen, issues, excluded = {}, {}, [], []\n    for split in (\'train\', \'val\', \'test\'):\n        rows = []\n        for image in sorted((dataset/\'images\'/split).glob(\'*\')):\n            if image.suffix.lower() not in (\'.jpg\', \'.png\', \'.jpeg\'):\n                continue\n            label = dataset/\'labels\'/split/(image.stem+\'.txt\')\n            if not label.is_file():\n                issues.append(f\'{split}/{image.name}: missing label\')\n                continue\n            boxes = []\n            for line in label.read_text().splitlines():\n                if not line.strip():\n                    continue\n                values = list(map(float, line.split()))\n                if len(values) != 5 or values[0] != 0 or not all(np.isfinite(values)) or not all(0 <= v <= 1 for v in values[1:]) or min(values[3:]) <= 0:\n                    issues.append(f\'{split}/{image.name}: invalid label\')\n                    break\n                boxes.append(values)\n            unique = {tuple(box) for box in boxes}\n            if len(boxes) > 1 and (len(boxes)-len(unique))/len(boxes) > .05:\n                issues.append(f\'{split}/{image.name}: repeated boxes indicate corrupt annotation conversion\')\n            digest = hashlib.sha256(image.read_bytes()).hexdigest()\n            if digest in seen:\n                excluded.append(f\'{split}/{image.name}: duplicate image of {seen[digest]}\')\n                continue\n            seen[digest] = f\'{split}/{image.name}\'\n            rows.append(dict(path=str(image.resolve()), count=len(boxes), sha256=digest))\n        splits[split] = rows\n    if issues:\n        raise ValueError(\'Dataset audit failed: \'+ \'; \'.join(issues[:12]))\n    if len(splits[\'val\']) < 112:\n        raise ValueError(\'Need at least 112 validation images for disjoint tuning/comparison subsets.\')\n    splits[\'excluded_duplicates\']=excluded\n    return splits\n\n\ndef evaluate(weights, rows, thresholds, imgsz=640, device=\'cpu\'):\n    model = YOLO(str(weights))\n    samples = []\n    start = time.perf_counter()\n    for i, row in enumerate(rows):\n        frame = cv2.imread(row[\'path\'])\n        if frame is None:\n            raise ValueError(\'Unreadable evaluation image: \'+row[\'path\'])\n        detections = detect_heads_in_frame(model, frame, conf=min(thresholds), iou=.5, imgsz=imgsz,\n                                          device=device, class_ids=[0], native_head_class=True)\n        label=Path(row[\'path\']).parent.parent.parent/\'labels\'/Path(row[\'path\']).parent.name/(Path(row[\'path\']).stem+\'.txt\')\n        truth=[]\n        h,w=frame.shape[:2]\n        for line in label.read_text().splitlines():\n            if line.strip():\n                _,x,y,bw,bh=map(float,line.split()); truth.append([(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h])\n        hits={}\n        for threshold in thresholds:\n            matched=set(); tp=0\n            for d in sorted((d for d in detections if d.confidence>=threshold),key=lambda d:-d.confidence):\n                best_i=None; best_iou=.5\n                for j,(x1,y1,x2,y2) in enumerate(truth):\n                    if j in matched: continue\n                    intersection=max(0,min(x2,d.x2)-max(x1,d.x1))*max(0,min(y2,d.y2)-max(y1,d.y1))\n                    union=(x2-x1)*(y2-y1)+(d.x2-d.x1)*(d.y2-d.y1)-intersection\n                    iou=intersection/union if union else 0\n                    if iou>=best_iou: best_iou=iou; best_i=j\n                if best_i is not None: matched.add(best_i); tp+=1\n            hits[str(threshold)]=tp\n        samples.append(dict(image=Path(row[\'path\']).name, truth=row[\'count\'],\n                            true_positives=hits,\n                            predictions={str(t): sum(d.confidence >= t for d in detections) for t in thresholds}))\n        if (i+1) % 8 == 0:\n            print(f\'Evaluated {i+1}/{len(rows)}: {Path(weights).name}\', flush=True)\n    metrics = {}\n    for threshold in thresholds:\n        errors = np.array([r[\'predictions\'][str(threshold)] - r[\'truth\'] for r in samples])\n        tp=sum(r[\'true_positives\'][str(threshold)] for r in samples)\n        predicted=sum(r[\'predictions\'][str(threshold)] for r in samples); truth=sum(r[\'truth\'] for r in samples)\n        metrics[str(threshold)] = dict(mae=float(abs(errors).mean()), rmse=float(np.sqrt((errors**2).mean())),\n                                      bias=float(errors.mean()), images=len(samples),precision=tp/max(1,predicted),recall=tp/max(1,truth))\n    return dict(metrics=metrics, samples=samples, seconds=time.perf_counter()-start, imgsz=imgsz)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--dataset\', type=Path, required=True)\n    parser.add_argument(\'--epochs\', type=int, default=3)\n    parser.add_argument(\'--train-images\', type=int, default=256)\n    parser.add_argument(\'--name\', default=\'campus_pilot_v1\')\n    parser.add_argument(\'--device\',default=\'cpu\')\n    parser.add_argument(\'--batch\',type=int,default=2)\n    parser.add_argument(\'--freeze\',type=int,default=22)\n    parser.add_argument(\'--imgsz\',type=int,default=640)\n    parser.add_argument(\'--val-images\',type=int,default=48)\n    parser.add_argument(\'--test-images\',type=int,default=64)\n    parser.add_argument(\'--patience\',type=int,default=8)\n    args = parser.parse_args()\n    torch.set_num_threads(4)\n    if args.device!=\'cpu\' and not torch.cuda.is_available():\n        raise RuntimeError(\'A CUDA GPU is required for this run. Select a GPU runtime before continuing.\')\n    experiment = ROOT/\'runs\'/\'experiments\'/args.name\n    experiment.mkdir(parents=True, exist_ok=False)\n    baseline = ROOT/\'models\'/\'head_detector.pt\'\n    data = audit(args.dataset)\n    rng = random.Random(20261027)\n    rng.shuffle(data[\'train\']); rng.shuffle(data[\'val\'])\n    train = data[\'train\'][:args.train_images] if args.train_images else data[\'train\']\n    tune = data[\'val\'][:args.val_images] if args.val_images else data[\'val\']\n    rng.shuffle(data[\'test\'])\n    holdout = (data[\'test\'][:args.test_images] if args.test_images else data[\'test\']) if data[\'test\'] else data[\'val\'][len(tune):len(tune)+64]\n    if not holdout: raise ValueError(\'A separate comparison split is required.\')\n    manifest = dict(seed=20261027, train=train, tune=tune, holdout=holdout,\n                    available={k:len(v) for k,v in data.items()}, excluded_duplicates=data[\'excluded_duplicates\'], baseline_sha256=hashlib.sha256(baseline.read_bytes()).hexdigest(),\n                    limitation=\'Existing checkpoint training exposure unknown; not an independently unseen test set.\')\n    atomic_json(experiment/\'manifest.json\', manifest)\n    for name, rows in [(\'train\',train), (\'val\',tune)]:\n        (experiment/f\'{name}.txt\').write_text(\'\\n\'.join(row[\'path\'] for row in rows),encoding=\'utf-8\')\n    dataset_yaml=experiment/\'data.yaml\'\n    dataset_yaml.write_text(yaml.safe_dump(dict(train=str(experiment/\'train.txt\'),val=str(experiment/\'val.txt\'),names={0:\'head\'},nc=1)),encoding=\'utf-8\')\n    atomic_json(experiment/\'status.json\', dict(stage=\'training\', epochs=args.epochs, train_images=len(train)))\n    model = YOLO(str(baseline))\n    model.train(data=str(dataset_yaml), epochs=args.epochs, imgsz=args.imgsz, batch=args.batch, workers=2 if args.device!=\'cpu\' else 0,\n                device=args.device, freeze=args.freeze, optimizer=\'AdamW\', lr0=.0001, lrf=.1,\n                mosaic=0, mixup=0, close_mosaic=0, degrees=0, translate=.05, scale=.15,\n                fliplr=.5, seed=20261027, deterministic=True, amp=args.device!=\'cpu\',\n                project=str(experiment), name=\'training\', plots=False, save=True, cache=False,\n                max_det=2000, patience=args.patience, save_period=5)\n    candidate=experiment/\'training\'/\'weights\'/\'best.pt\'\n    thresholds=[.15,.25,.35,.45]\n    atomic_json(experiment/\'status.json\',dict(stage=\'threshold_tuning\'))\n    baseline_tune=evaluate(baseline,tune,thresholds,device=args.device)\n    candidate_tune=evaluate(candidate,tune,thresholds,device=args.device)\n    baseline_conf=float(min(baseline_tune[\'metrics\'],key=lambda t:baseline_tune[\'metrics\'][t][\'mae\']))\n    candidate_conf=float(min(candidate_tune[\'metrics\'],key=lambda t:candidate_tune[\'metrics\'][t][\'mae\']))\n    atomic_json(experiment/\'status.json\',dict(stage=\'heldout_comparison\'))\n    baseline_test=evaluate(baseline,holdout,[baseline_conf],device=args.device)\n    candidate_test=evaluate(candidate,holdout,[candidate_conf],device=args.device)\n    old=baseline_test[\'metrics\'][str(baseline_conf)]; new=candidate_test[\'metrics\'][str(candidate_conf)]\n    eligible=new[\'mae\'] < old[\'mae\']*.98 and new[\'rmse\'] <= old[\'rmse\'] and new[\'precision\']>=old[\'precision\']-.02 and new[\'recall\']>=old[\'recall\']-.02\n    comparison=dict(status=\'complete\',eligible_for_promotion=eligible,baseline_confidence=baseline_conf,\n                    candidate_confidence=candidate_conf,baseline=old,candidate=new,\n                    baseline_tuning=baseline_tune,candidate_tuning=candidate_tune,\n                    baseline_test=baseline_test,candidate_test=candidate_test,\n                    candidate_path=str(candidate),train_images=len(train),epochs=args.epochs,\n                    limitation=manifest[\'limitation\'])\n    atomic_json(experiment/\'comparison.json\', comparison)\n    atomic_json(experiment/\'status.json\',dict(stage=\'complete\',eligible_for_promotion=eligible))\n    print(json.dumps({k:comparison[k] for k in (\'eligible_for_promotion\',\'baseline\',\'candidate\')},indent=2),flush=True)\n\nif __name__==\'__main__\':\n    main()\n'}
for name,content in SOURCES.items():
    path=ROOT/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content,encoding='utf-8')
(ROOT/'models').mkdir(exist_ok=True)
baseline=ROOT/'models/head_detector.pt'
if not baseline.exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/Tishal68/AI-Crowd-Management-System/af246a2/models/head_detector.pt',baseline)
assert hashlib.sha256(baseline.read_bytes()).hexdigest()=='667b4a6da2b8739b4a11e2070e281506e6303de76589cb2832d80c261b99004c','Baseline checksum mismatch'
print('Pinned baseline and source code ready.')


In [ ]:
import gdown
archive=ROOT/'SCUT_HEAD_Part_B.zip'
if not archive.exists():
    gdown.download(id='1LZ_KlTPStDEcqycfqUkDkqQ-aNMMC3cl',output=str(archive),quiet=False)
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='02afedc58065aca9cb5f28333963a818e37564d75d8420f9bcb94d1e6c340a98','Dataset checksum mismatch'
dataset=ROOT/'scut_head'
if not (dataset/'provenance.json').exists():
    if dataset.exists(): raise RuntimeError('Partial conversion exists. Use a new dataset folder or a fresh runtime.')
    subprocess.run([sys.executable,'scripts/prepare_scut.py',str(archive),str(dataset)],check=True)
print((dataset/'provenance.json').read_text())


## Train and compare
Use all deduplicated official splits. Training stops at 30 epochs or after 8 non-improving epochs. If GPU memory runs out, change BATCH to 4 and rerun with a new run name. Exact duplicates are excluded; near-duplicates may remain. Evaluation uses the application's 640-pixel inference setting for both models.


In [ ]:
from datetime import datetime,timezone
EPOCHS=30
BATCH=8
RUN_NAME='scut_gpu_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
command=[sys.executable,'-u','scripts/model_experiment.py','--dataset',str(dataset),
         '--name',RUN_NAME,'--epochs',str(EPOCHS),'--train-images','0','--val-images','0',
         '--test-images','0','--device','0','--batch',str(BATCH),'--freeze','0','--imgsz','768','--patience','8']
with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for output_line in process.stdout:
        print(output_line,end='',flush=True)
    if process.wait(): raise RuntimeError('Training or evaluation failed. Inspect the output above.')


In [ ]:
import zipfile
import pandas as pd
from IPython.display import display
experiment=ROOT/'runs/experiments'/RUN_NAME
comparison=json.loads((experiment/'comparison.json').read_text())
display(pd.DataFrame({'Previous model':comparison['baseline'],'Trained candidate':comparison['candidate']}))
print('Eligible for replacement:',comparison['eligible_for_promotion'])
print(comparison['limitation'])
package=ROOT/(RUN_NAME+'_results.zip')
with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for relative in ['comparison.json','manifest.json','training/weights/best.pt','training/weights/last.pt','training/results.csv','training/args.yaml']:
        path=experiment/relative
        if path.exists(): z.write(path,relative)
    z.write(dataset/'provenance.json','dataset_provenance.json')
print('Keep this download:',package.name)
from google.colab import files
files.download(str(package))
